In [1]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — LOAD ROUND 5 DATA
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROUND = 5
DAYS = [2, 3, 4]

DATA_DIR = Path("Data/round5")

ALGO_PRODUCTS = [
    # Galaxy Sounds Recorders
    "GALAXY_SOUNDS_DARK_MATTER",
    "GALAXY_SOUNDS_BLACK_HOLES",
    "GALAXY_SOUNDS_PLANETARY_RINGS",
    "GALAXY_SOUNDS_SOLAR_WINDS",
    "GALAXY_SOUNDS_SOLAR_FLAMES",

    # Vertical Sleeping Pods
    "SLEEP_POD_SUEDE",
    "SLEEP_POD_LAMB_WOOL",
    "SLEEP_POD_POLYESTER",
    "SLEEP_POD_NYLON",
    "SLEEP_POD_COTTON",

    # Organic Microchips
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",

    # Purification Pebbles
    "PEBBLES_XS",
    "PEBBLES_S",
    "PEBBLES_M",
    "PEBBLES_L",
    "PEBBLES_XL",

    # Domestic Robots
    "ROBOT_VACUUMING",
    "ROBOT_MOPPING",
    "ROBOT_DISHES",
    "ROBOT_LAUNDRY",
    "ROBOT_IRONING",

    # UV-Visors
    "UV_VISOR_YELLOW",
    "UV_VISOR_AMBER",
    "UV_VISOR_ORANGE",
    "UV_VISOR_RED",
    "UV_VISOR_MAGENTA",

    # Instant Translators
    "TRANSLATOR_SPACE_GRAY",
    "TRANSLATOR_ASTRO_BLACK",
    "TRANSLATOR_ECLIPSE_CHARCOAL",
    "TRANSLATOR_GRAPHITE_MIST",
    "TRANSLATOR_VOID_BLUE",

    # Construction Panels
    "PANEL_1X2",
    "PANEL_2X2",
    "PANEL_1X4",
    "PANEL_2X4",
    "PANEL_4X4",

    # Liquid Breath Oxygen Shakes
    "OXYGEN_SHAKE_MORNING_BREATH",
    "OXYGEN_SHAKE_EVENING_BREATH",
    "OXYGEN_SHAKE_MINT",
    "OXYGEN_SHAKE_CHOCOLATE",
    "OXYGEN_SHAKE_GARLIC",

    # Protein Snack Packs
    "SNACKPACK_CHOCOLATE",
    "SNACKPACK_VANILLA",
    "SNACKPACK_PISTACHIO",
    "SNACKPACK_STRAWBERRY",
    "SNACKPACK_RASPBERRY",
]

POSITION_LIMITS = {product: 10 for product in ALGO_PRODUCTS}

prices_parts = []
trades_parts = []

for day in DAYS:
    price_path = DATA_DIR / f"prices_round_{ROUND}_day_{day}.csv"
    trade_path = DATA_DIR / f"trades_round_{ROUND}_day_{day}.csv"

    p = pd.read_csv(price_path, sep=";")
    t = pd.read_csv(trade_path, sep=";")

    p["file_day"] = day
    t["file_day"] = day

    prices_parts.append(p)
    trades_parts.append(t)

prices = pd.concat(prices_parts, ignore_index=True)
trades = pd.concat(trades_parts, ignore_index=True)

# Standardise trade product column name.
if "symbol" in trades.columns and "product" not in trades.columns:
    trades = trades.rename(columns={"symbol": "product"})

# Keep only valid Round 5 algorithmic products.
prices = prices[prices["product"].isin(ALGO_PRODUCTS)].copy()
trades = trades[trades["product"].isin(ALGO_PRODUCTS)].copy()

# Useful global time index across days.
# Assumes timestamp resets each day.
min_day = min(DAYS)
prices["global_ts"] = (prices["file_day"] - min_day) * 1_000_000 + prices["timestamp"]
trades["global_ts"] = (trades["file_day"] - min_day) * 1_000_000 + trades["timestamp"]

prices = prices.sort_values(["product", "global_ts"]).reset_index(drop=True)
trades = trades.sort_values(["product", "global_ts"]).reset_index(drop=True)

# Basic sanity checks.
price_products = sorted(prices["product"].unique())
trade_products = sorted(trades["product"].unique())

missing_in_prices = sorted(set(ALGO_PRODUCTS) - set(price_products))
missing_in_trades = sorted(set(ALGO_PRODUCTS) - set(trade_products))

print("prices shape:", prices.shape)
print("trades shape:", trades.shape)
print()
print("Price products:", price_products)
print("Trade products:", trade_products)
print()
print("Missing in prices:", missing_in_prices)
print("Missing in trades:", missing_in_trades)
print()
print("Price days:", sorted(prices["file_day"].unique()))
print("Trade days:", sorted(trades["file_day"].unique()))
print()
print("Position limits:", POSITION_LIMITS)

display(prices.head())
display(trades.head())

prices shape: (1500000, 19)
trades shape: (35385, 9)

Price products: ['GALAXY_SOUNDS_BLACK_HOLES', 'GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_FLAMES', 'GALAXY_SOUNDS_SOLAR_WINDS', 'MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_RECTANGLE', 'MICROCHIP_SQUARE', 'MICROCHIP_TRIANGLE', 'OXYGEN_SHAKE_CHOCOLATE', 'OXYGEN_SHAKE_EVENING_BREATH', 'OXYGEN_SHAKE_GARLIC', 'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_MORNING_BREATH', 'PANEL_1X2', 'PANEL_1X4', 'PANEL_2X2', 'PANEL_2X4', 'PANEL_4X4', 'PEBBLES_L', 'PEBBLES_M', 'PEBBLES_S', 'PEBBLES_XL', 'PEBBLES_XS', 'ROBOT_DISHES', 'ROBOT_IRONING', 'ROBOT_LAUNDRY', 'ROBOT_MOPPING', 'ROBOT_VACUUMING', 'SLEEP_POD_COTTON', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_NYLON', 'SLEEP_POD_POLYESTER', 'SLEEP_POD_SUEDE', 'SNACKPACK_CHOCOLATE', 'SNACKPACK_PISTACHIO', 'SNACKPACK_RASPBERRY', 'SNACKPACK_STRAWBERRY', 'SNACKPACK_VANILLA', 'TRANSLATOR_ASTRO_BLACK', 'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST', 'TRANSLATOR_SPACE_GRAY'

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,file_day,global_ts
0,2,0,GALAXY_SOUNDS_BLACK_HOLES,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,2,0
1,2,100,GALAXY_SOUNDS_BLACK_HOLES,10001,18,10000.0,25.0,NaN,NaN,10014,18,10016.0,25.0,NaN,NaN,10007.5,0.0,2,100
2,2,200,GALAXY_SOUNDS_BLACK_HOLES,9996,19,9995.0,31.0,NaN,NaN,10009,19,10011.0,31.0,NaN,NaN,10002.5,0.0,2,200
3,2,300,GALAXY_SOUNDS_BLACK_HOLES,9994,25,9993.0,33.0,NaN,NaN,10007,25,10009.0,33.0,NaN,NaN,10000.5,0.0,2,300
4,2,400,GALAXY_SOUNDS_BLACK_HOLES,9999,14,9997.0,32.0,NaN,NaN,10012,14,10013.0,32.0,NaN,NaN,10005.5,0.0,2,400


,timestamp,buyer,seller,product,currency,price,quantity,file_day,global_ts
0,1700,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9969.0,4,2,1700
1,14500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9749.0,1,2,14500
2,15100,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9764.0,2,2,15100
3,26500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9656.0,4,2,26500
4,36400,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9675.0,4,2,36400


In [3]:
# =========================
# UV VISORS DISCOVERY + STRICT BACKTEST NOTEBOOK
# Assumes you already have: prices = market dataframe
# =========================

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations, permutations, product
import time

OUTDIR = Path("analysis_outputs/uv_visors")
OUTDIR.mkdir(parents=True, exist_ok=True)

VISORS = [
    "UV_VISOR_YELLOW",
    "UV_VISOR_AMBER",
    "UV_VISOR_ORANGE",
    "UV_VISOR_RED",
    "UV_VISOR_MAGENTA",
]

COLOR_SHORT = {
    "UV_VISOR_YELLOW": "YELLOW",
    "UV_VISOR_AMBER": "AMBER",
    "UV_VISOR_ORANGE": "ORANGE",
    "UV_VISOR_RED": "RED",
    "UV_VISOR_MAGENTA": "MAGENTA",
}

# Ordered colour spectrum assumption:
# YELLOW -> AMBER -> ORANGE -> RED -> MAGENTA
COLOR_ORDER = VISORS


# =========================
# 1) Robust dataframe parsing
# =========================

def pick_col(df, options, required=True):
    for c in options:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not find any of columns: {options}")
    return None


def prepare_prices_df(prices):
    df = prices.copy()

    product_col = pick_col(df, ["product", "symbol", "asset", "instrument"])
    day_col = pick_col(df, ["day", "trading_day", "date"])
    ts_col = pick_col(df, ["timestamp", "time", "ts"])

    bid_col = pick_col(
        df,
        ["bid_price_1", "bid1", "best_bid", "bid_price", "bid"],
        required=False,
    )
    ask_col = pick_col(
        df,
        ["ask_price_1", "ask1", "best_ask", "ask_price", "ask"],
        required=False,
    )
    mid_col = pick_col(
        df,
        ["mid_price", "mid", "fair", "price"],
        required=False,
    )

    if mid_col is None:
        if bid_col is None or ask_col is None:
            raise ValueError("Need either mid column, or bid+ask columns.")
        df["_mid"] = (df[bid_col] + df[ask_col]) / 2
        mid_col = "_mid"

    if bid_col is None:
        df["_bid"] = df[mid_col]
        bid_col = "_bid"
    if ask_col is None:
        df["_ask"] = df[mid_col]
        ask_col = "_ask"

    return df, product_col, day_col, ts_col, bid_col, ask_col, mid_col


def make_day_panels(prices, products):
    df, product_col, day_col, ts_col, bid_col, ask_col, mid_col = prepare_prices_df(prices)
    df = df[df[product_col].isin(products)].copy()

    missing = sorted(set(products) - set(df[product_col].unique()))
    if missing:
        raise ValueError(f"Missing visor products in dataframe: {missing}")

    panels = {}
    for day, d in df.groupby(day_col):
        mid = d.pivot(index=ts_col, columns=product_col, values=mid_col).sort_index()
        bid = d.pivot(index=ts_col, columns=product_col, values=bid_col).sort_index()
        ask = d.pivot(index=ts_col, columns=product_col, values=ask_col).sort_index()

        # Align columns/order and fill very small data gaps if any.
        mid = mid.reindex(columns=products).ffill().bfill()
        bid = bid.reindex(columns=products).ffill().bfill()
        ask = ask.reindex(columns=products).ffill().bfill()

        panels[int(day)] = {"mid": mid, "bid": bid, "ask": ask}

    return panels


panels = make_day_panels(prices, VISORS)
DAYS = sorted(panels.keys())

print("Days:", DAYS)
for day in DAYS:
    print(
        f"Day {day}: mid={panels[day]['mid'].shape}, "
        f"bid={panels[day]['bid'].shape}, ask={panels[day]['ask'].shape}"
    )

Days: [2, 3, 4]
Day 2: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 3: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 4: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)


In [5]:
# =========================
# 2) Helpers: z-score, q generation, execution PnL
# =========================

def rolling_z(s, window):
    s = pd.Series(s).replace([np.inf, -np.inf], np.nan)
    mu = s.rolling(window, min_periods=window).mean()
    sd = s.rolling(window, min_periods=window).std(ddof=0)
    return ((s - mu) / sd).replace([np.inf, -np.inf], np.nan)


def q_from_weights(weights, gross=60, max_abs=80):
    """
    Converts arbitrary signal/position weights into integer trade quantities.
    Sign convention:
      weights = desired position when signal is LOW for mean-reversion.
    """
    w = np.asarray(weights, dtype=float)
    if np.allclose(w, 0):
        return np.zeros_like(w, dtype=int)

    q = w / np.sum(np.abs(w)) * gross
    q = np.rint(q).astype(int)

    # Avoid losing tiny but intentional legs after rounding.
    for i, wi in enumerate(w):
        if wi != 0 and q[i] == 0:
            q[i] = int(np.sign(wi))

    if np.max(np.abs(q)) > max_abs:
        scale = max_abs / np.max(np.abs(q))
        q = np.rint(q * scale).astype(int)

    return q


def exec_pnl(pos, entry_i, exit_i, bid, ask):
    """
    Conservative execution:
      Long: buy at entry ask, sell at exit bid.
      Short: sell at entry bid, buy at exit ask.
    """
    pos = np.asarray(pos, dtype=float)

    entry_bid = bid.iloc[entry_i].values.astype(float)
    entry_ask = ask.iloc[entry_i].values.astype(float)
    exit_bid = bid.iloc[exit_i].values.astype(float)
    exit_ask = ask.iloc[exit_i].values.astype(float)

    long = pos > 0
    short = pos < 0

    pnl = 0.0
    pnl += np.sum(pos[long] * (exit_bid[long] - entry_ask[long]))
    pnl += np.sum((-pos[short]) * (entry_bid[short] - exit_ask[short]))
    return float(pnl)


def summarise_day_rows(rows):
    if not rows:
        return {
            "trade_count": 0,
            "day_pnl": 0.0,
            "hit_rate": np.nan,
            "avg_trade_pnl": np.nan,
            "median_trade_pnl": np.nan,
        }

    pnls = np.array([r["exec_pnl"] for r in rows], dtype=float)
    return {
        "trade_count": len(pnls),
        "day_pnl": float(pnls.sum()),
        "hit_rate": float(np.mean(pnls > 0)),
        "avg_trade_pnl": float(np.mean(pnls)),
        "median_trade_pnl": float(np.median(pnls)),
    }


def robust_score(total_pnl, min_day_pnl, mean_hit_rate, one_day_dependency):
    if total_pnl == 0 or not np.isfinite(total_pnl):
        return -np.inf
    dep_penalty = max(0.0, one_day_dependency - 0.50) * abs(total_pnl) * 0.5
    return float(total_pnl + 2.0 * min_day_pnl + 5000.0 * mean_hit_rate - dep_penalty)

In [6]:
# =========================
# 3) Candidate generation
# Includes pairs, lead-lag, baskets, colour gradients, and colour mixing
# =========================

def idx(product):
    return VISORS.index(product)


def vector_from_dict(d):
    v = np.zeros(len(VISORS), dtype=float)
    for k, val in d.items():
        v[idx(k)] = val
    return v


def linear_signal(weights):
    weights = np.asarray(weights, dtype=float)
    def f(mid):
        return mid.dot(weights)
    return f


def pair_spread_signal(a, b):
    return lambda mid: mid[a] - mid[b]


def pair_logratio_signal(a, b):
    return lambda mid: np.log(mid[a] / mid[b])


def pair_ratio_signal(a, b):
    return lambda mid: mid[a] / mid[b]


def leader_return_signal(leader, lag):
    return lambda mid: np.log(mid[leader] / mid[leader].shift(lag))


def static_poly_resid_signal(y, x, degree):
    """
    Discovery-only: fits y ~ poly(x) using the full day.
    Useful to detect nonlinear relationships, but don't trust as final
    unless later converted to rolling/online form.
    """
    def f(mid):
        xx = mid[x].astype(float)
        yy = mid[y].astype(float)
        ok = xx.notna() & yy.notna()
        if ok.sum() < degree + 2:
            return pd.Series(np.nan, index=mid.index)
        coefs = np.polyfit(xx[ok], yy[ok], degree)
        pred = np.polyval(coefs, xx)
        return yy - pred
    return f


def add_candidate(cands, name, family, signal_type, signal_func, pos_low_weights, note):
    cands.append({
        "candidate": name,
        "family": family,
        "signal_type": signal_type,
        "signal_func": signal_func,
        "pos_low_weights": np.asarray(pos_low_weights, dtype=float),
        "note": note,
    })


def make_uv_visor_candidates(include_static_poly=True):
    cands = []

    # ---- Single product own mean-reversion / trend
    for p in VISORS:
        w = vector_from_dict({p: 1})
        short = COLOR_SHORT[p]

        add_candidate(
            cands,
            f"{short}_own_meanrev",
            "single",
            "own_price",
            lambda mid, p=p: mid[p],
            w,
            f"{short} own price mean-reversion",
        )

        add_candidate(
            cands,
            f"{short}_own_trend",
            "single",
            "own_price",
            lambda mid, p=p: mid[p],
            -w,
            f"{short} own price trend/breakout",
        )

    # ---- Pairwise spreads / ratios / logratios, both meanrev and trend
    for a, b in combinations(VISORS, 2):
        sa, sb = COLOR_SHORT[a], COLOR_SHORT[b]
        w = vector_from_dict({a: 1, b: -1})

        for signal_type, sig_fn in [
            ("spread", pair_spread_signal(a, b)),
            ("logratio", pair_logratio_signal(a, b)),
            ("ratio", pair_ratio_signal(a, b)),
        ]:
            add_candidate(
                cands,
                f"{sa}_vs_{sb}_{signal_type}_mr",
                "pairwise",
                signal_type,
                sig_fn,
                w,
                f"{sa} - {sb} mean-reversion",
            )
            add_candidate(
                cands,
                f"{sa}_vs_{sb}_{signal_type}_trend",
                "pairwise",
                signal_type,
                sig_fn,
                -w,
                f"{sa} - {sb} trend",
            )

        if include_static_poly:
            for deg in [2, 3]:
                add_candidate(
                    cands,
                    f"{sa}_resid_vs_{sb}_poly{deg}_mr_DISCOVERY",
                    "pairwise_nonlinear_discovery",
                    f"poly{deg}_resid",
                    static_poly_resid_signal(a, b, deg),
                    w,
                    f"{sa} residual versus degree-{deg} function of {sb}; discovery-only",
                )

                add_candidate(
                    cands,
                    f"{sb}_resid_vs_{sa}_poly{deg}_mr_DISCOVERY",
                    "pairwise_nonlinear_discovery",
                    f"poly{deg}_resid",
                    static_poly_resid_signal(b, a, deg),
                    -w,
                    f"{sb} residual versus degree-{deg} function of {sa}; discovery-only",
                )

    # ---- Lead-lag: every leader/follower, follow and inverse
    for leader, follower in permutations(VISORS, 2):
        sl, sf = COLOR_SHORT[leader], COLOR_SHORT[follower]
        follower_w = vector_from_dict({follower: 1})

        for lag in [50, 100, 250, 500, 1000]:
            sig = leader_return_signal(leader, lag)

            # follow: leader down => short follower, leader up => long follower
            add_candidate(
                cands,
                f"lead_{sl}_to_{sf}_follow_lag{lag}",
                "lead_lag",
                "leader_return_follow",
                sig,
                -follower_w,
                f"{sf} follows {sl} return over lag {lag}",
            )

            # inverse: leader down => long follower, leader up => short follower
            add_candidate(
                cands,
                f"lead_{sl}_to_{sf}_inverse_lag{lag}",
                "lead_lag",
                "leader_return_inverse",
                sig,
                follower_w,
                f"{sf} inversely follows {sl} return over lag {lag}",
            )

    # ---- Ordered colour spectrum baskets
    Y, A, O, R, M = VISORS

    basket_defs = {
        "spectrum_gradient_2_1_0_1_2": {Y: -2, A: -1, O: 0, R: 1, M: 2},
        "spectrum_gradient_1_1_0_1_1": {Y: -1, A: -1, O: 0, R: 1, M: 1},
        "warm_vs_cool": {Y: -1, A: -1, O: 1, R: 1, M: 1},
        "hot_vs_nonhot": {Y: -1, A: -1, O: -1, R: 1.5, M: 1.5},
        "red_extreme_vs_rest": {Y: -0.25, A: -0.25, O: -0.25, R: 1, M: -0.25},
        "magenta_extreme_vs_rest": {Y: -0.25, A: -0.25, O: -0.25, R: -0.25, M: 1},
        "orange_red_vs_others": {Y: -1, A: -1, O: 1.5, R: 1.5, M: -1},
        "yellow_amber_vs_red_magenta": {Y: 1, A: 1, O: 0, R: -1, M: -1},
    }

    for name, wd in basket_defs.items():
        w = vector_from_dict(wd)
        add_candidate(
            cands,
            f"basket_{name}_mr",
            "colour_basket",
            "linear_basket",
            linear_signal(w),
            w,
            f"Colour basket mean-reversion: {name}",
        )
        add_candidate(
            cands,
            f"basket_{name}_trend",
            "colour_basket",
            "linear_basket",
            linear_signal(w),
            -w,
            f"Colour basket trend: {name}",
        )

    # ---- Adjacent neighbour spreads along colour spectrum
    adjacent = [(Y, A), (A, O), (O, R), (R, M)]
    for a, b in adjacent:
        sa, sb = COLOR_SHORT[a], COLOR_SHORT[b]
        w = vector_from_dict({a: 1, b: -1})
        add_candidate(
            cands,
            f"adjacent_{sa}_vs_{sb}_spread_mr",
            "colour_adjacent",
            "spread",
            pair_spread_signal(a, b),
            w,
            f"Adjacent colour spread {sa}-{sb} mean-reversion",
        )

    # ---- Explicit colour mixing / blend residuals
    # Signal: target - weighted blend(source1, source2)
    # Low signal => target cheap vs blend => long target, short blend legs.
    mix_specs = []

    # Local spectrum blends
    mix_specs += [
        ("AMBER_blend_YELLOW_ORANGE_50_50", A, Y, O, 0.50),
        ("ORANGE_blend_AMBER_RED_50_50", O, A, R, 0.50),
        ("RED_blend_ORANGE_MAGENTA_50_50", R, O, M, 0.50),
    ]

    # Wider colour-mixing hypotheses
    for alpha in [0.25, 0.50, 0.75]:
        mix_specs += [
            (f"ORANGE_blend_YELLOW_RED_{int(alpha*100)}_{int((1-alpha)*100)}", O, Y, R, alpha),
            (f"RED_blend_YELLOW_MAGENTA_{int(alpha*100)}_{int((1-alpha)*100)}", R, Y, M, alpha),
            (f"AMBER_blend_YELLOW_RED_{int(alpha*100)}_{int((1-alpha)*100)}", A, Y, R, alpha),
            (f"ORANGE_blend_YELLOW_MAGENTA_{int(alpha*100)}_{int((1-alpha)*100)}", O, Y, M, alpha),
        ]

    for name, target, s1, s2, alpha in mix_specs:
        w = vector_from_dict({
            target: 1,
            s1: -alpha,
            s2: -(1 - alpha),
        })
        add_candidate(
            cands,
            f"mix_{name}_mr",
            "colour_mixing",
            "blend_residual",
            linear_signal(w),
            w,
            f"{COLOR_SHORT[target]} residual versus blend of {COLOR_SHORT[s1]} and {COLOR_SHORT[s2]}",
        )

    # ---- Curvature / second-difference relationships along the colour line
    curvature_specs = [
        ("curvature_YELLOW_AMBER_ORANGE", {Y: 1, A: -2, O: 1}),
        ("curvature_AMBER_ORANGE_RED", {A: 1, O: -2, R: 1}),
        ("curvature_ORANGE_RED_MAGENTA", {O: 1, R: -2, M: 1}),
        ("curvature_YELLOW_ORANGE_RED", {Y: 1, O: -2, R: 1}),
        ("curvature_AMBER_RED_MAGENTA", {A: 1, R: -2, M: 1}),
    ]

    for name, wd in curvature_specs:
        w = vector_from_dict(wd)
        add_candidate(
            cands,
            f"{name}_mr",
            "colour_curvature",
            "second_difference",
            linear_signal(w),
            w,
            f"Colour curvature mean-reversion: {name}",
        )

    return cands


candidates = make_uv_visor_candidates(include_static_poly=True)
print("Candidates generated:", len(candidates))
print(pd.Series([c["family"] for c in candidates]).value_counts())

Candidates generated: 350
lead_lag                        200
pairwise                         60
pairwise_nonlinear_discovery     40
colour_basket                    16
colour_mixing                    15
single                           10
colour_curvature                  5
colour_adjacent                   4
Name: count, dtype: int64


In [7]:
# =========================
# 4) Lead-lag diagnostics table
# Not a backtest, just correlation/edge discovery.
# =========================

def run_leadlag_diagnostics(panels, products, lags=(50, 100, 250, 500, 1000), horizons=(50, 100, 250, 500, 1000)):
    rows = []

    for leader, follower in permutations(products, 2):
        for lag in lags:
            for h in horizons:
                corrs = []
                edges = []

                for day, panel in panels.items():
                    mid = panel["mid"]
                    leader_ret = np.log(mid[leader] / mid[leader].shift(lag))
                    follower_future = mid[follower].shift(-h) - mid[follower]

                    z = rolling_z(leader_ret, max(250, lag * 2))
                    mask = z.abs() > 1.5

                    corr = leader_ret.corr(follower_future)
                    edge = np.nan
                    if mask.sum() > 10:
                        # Signed edge: if leader_ret positive, expected follower move;
                        # if negative, expected opposite sign.
                        edge = np.nanmean(np.sign(leader_ret[mask]) * follower_future[mask])

                    corrs.append(corr)
                    edges.append(edge)

                rows.append({
                    "leader": leader,
                    "follower": follower,
                    "lag": lag,
                    "future_horizon": h,
                    "mean_corr": float(np.nanmean(corrs)),
                    "mean_signed_edge": float(np.nanmean(edges)),
                    "abs_score": float(abs(np.nanmean(corrs)) * abs(np.nanmean(edges))),
                    "suggested_mode": "follow" if np.nanmean(corrs) > 0 else "inverse",
                })

    out = pd.DataFrame(rows).sort_values("abs_score", ascending=False)
    out.to_csv(OUTDIR / "uv_visors_leadlag_diagnostics.csv", index=False)
    return out


leadlag_diag = run_leadlag_diagnostics(panels, VISORS)
display(leadlag_diag.head(50))

,leader,follower,lag,future_horizon,mean_corr,mean_signed_edge,abs_score,suggested_mode
199,UV_VISOR_AMBER,UV_VISOR_MAGENTA,1000,1000,0.451211,218.437422,98.561429,follow
49,UV_VISOR_YELLOW,UV_VISOR_ORANGE,1000,1000,-0.439975,-97.643963,42.960894,inverse
318,UV_VISOR_RED,UV_VISOR_YELLOW,500,500,-0.327298,-108.636374,35.556429,inverse
494,UV_VISOR_MAGENTA,UV_VISOR_RED,500,1000,-0.263304,-95.814441,25.228362,inverse
319,UV_VISOR_RED,UV_VISOR_YELLOW,500,1000,-0.186026,-129.728806,24.132987,inverse
344,UV_VISOR_RED,UV_VISOR_AMBER,500,1000,0.250122,87.858101,21.975201,follow
398,UV_VISOR_RED,UV_VISOR_MAGENTA,1000,500,-0.169944,-108.121912,18.374695,inverse
394,UV_VISOR_RED,UV_VISOR_MAGENTA,500,1000,-0.127305,-139.870049,17.806143,inverse
317,UV_VISOR_RED,UV_VISOR_YELLOW,500,250,-0.293791,-60.018023,17.632756,inverse
198,UV_VISOR_AMBER,UV_VISOR_MAGENTA,1000,500,0.266334,63.350984,16.872509,follow


In [8]:
# =========================
# 5) Loose event scan
# Ranks broad families quickly before strict backtest.
# =========================

def event_scan_one_config(cand, window, threshold, horizon, gross):
    q_low = q_from_weights(cand["pos_low_weights"], gross=gross)

    day_rows = []
    all_trades = []

    for day, panel in panels.items():
        mid, bid, ask = panel["mid"], panel["bid"], panel["ask"]
        sig = cand["signal_func"](mid)
        z = rolling_z(sig, window)

        trades = []
        i = window

        while i < len(mid) - horizon:
            zi = z.iloc[i]
            if not np.isfinite(zi):
                i += 1
                continue

            if zi <= -threshold:
                pos = q_low
                side = 1
            elif zi >= threshold:
                pos = -q_low
                side = -1
            else:
                i += 1
                continue

            exit_i = i + horizon
            pnl = exec_pnl(pos, i, exit_i, bid, ask)

            trade = {
                "candidate": cand["candidate"],
                "family": cand["family"],
                "signal_type": cand["signal_type"],
                "note": cand["note"],
                "day": day,
                "window": window,
                "threshold": threshold,
                "horizon": horizon,
                "gross": gross,
                "entry_idx": i,
                "exit_idx": exit_i,
                "entry_ts": mid.index[i],
                "exit_ts": mid.index[exit_i],
                "side": side,
                "entry_z_seen": float(zi),
                "exec_pnl": pnl,
                "q_trade": q_low.tolist(),
                "position": pos.tolist(),
            }
            trades.append(trade)
            all_trades.append(trade)

            # Non-overlapping event scan.
            i = exit_i + 1

        s = summarise_day_rows(trades)
        s.update({
            "candidate": cand["candidate"],
            "family": cand["family"],
            "signal_type": cand["signal_type"],
            "note": cand["note"],
            "day": day,
            "window": window,
            "threshold": threshold,
            "horizon": horizon,
            "gross": gross,
            "q_trade": q_low.tolist(),
            "net_position": float(np.sum(q_low)),
            "gross_position": float(np.sum(np.abs(q_low))),
        })
        day_rows.append(s)

    day_df = pd.DataFrame(day_rows)
    total_pnl = day_df["day_pnl"].sum()
    total_trades = day_df["trade_count"].sum()
    active_days = int((day_df["trade_count"] > 0).sum())
    mean_hit = day_df["hit_rate"].mean()
    min_hit = day_df["hit_rate"].min()
    min_day = day_df["day_pnl"].min()
    max_day = day_df["day_pnl"].max()
    one_day_dep = abs(max_day) / abs(total_pnl) if total_pnl != 0 else np.inf

    summary = {
        "candidate": cand["candidate"],
        "family": cand["family"],
        "signal_type": cand["signal_type"],
        "note": cand["note"],
        "window": window,
        "threshold": threshold,
        "horizon": horizon,
        "gross": gross,
        "q_trade": q_low.tolist(),
        "net_position": float(np.sum(q_low)),
        "gross_position": float(np.sum(np.abs(q_low))),
        "days": len(DAYS),
        "active_days": active_days,
        "total_events": int(total_trades),
        "total_pnl": float(total_pnl),
        "mean_day_pnl": float(day_df["day_pnl"].mean()),
        "min_day_pnl": float(min_day),
        "max_day_pnl": float(max_day),
        "mean_hit_rate": float(mean_hit),
        "min_hit_rate": float(min_hit),
        "pnl_per_event": float(total_pnl / total_trades) if total_trades else np.nan,
        "one_day_dependency": float(one_day_dep),
    }
    summary["robust_pass"] = (
        active_days == len(DAYS)
        and total_trades >= 6
        and min_day > 0
        and mean_hit > 0.55
    )
    summary["robust_score"] = robust_score(total_pnl, min_day, mean_hit, one_day_dep)

    return summary, day_df, all_trades


EVENT_WINDOWS = [500, 1000, 2500, 4000, 5000]
EVENT_THRESHOLDS = [1.25, 1.5, 1.75, 2.0]
EVENT_HORIZONS = [250, 500, 1000, 1500, 2500]
EVENT_GROSS = 60

event_summaries = []
event_day_breakdowns = []
event_trades = []

configs = list(product(range(len(candidates)), EVENT_WINDOWS, EVENT_THRESHOLDS, EVENT_HORIZONS))
t0 = time.time()

for k, (ci, w, th, h) in enumerate(configs, start=1):
    cand = candidates[ci]

    if k == 1 or k % 500 == 0 or k == len(configs):
        print(f"[{time.time()-t0:7.2f}s] event {k}/{len(configs)}: {cand['candidate']}, w={w}, th={th}, h={h}")

    summary, day_df, trades = event_scan_one_config(cand, w, th, h, EVENT_GROSS)
    event_summaries.append(summary)
    event_day_breakdowns.append(day_df)
    event_trades.extend(trades)

event_summary = pd.DataFrame(event_summaries).sort_values("robust_score", ascending=False)
event_day_breakdown = pd.concat(event_day_breakdowns, ignore_index=True)
event_trades_df = pd.DataFrame(event_trades)

event_summary.to_csv(OUTDIR / "uv_visors_event_summary.csv", index=False)
event_summary[event_summary["robust_pass"]].to_csv(OUTDIR / "uv_visors_event_robust_only.csv", index=False)
event_day_breakdown.to_csv(OUTDIR / "uv_visors_event_day_breakdown.csv", index=False)
event_trades_df.to_csv(OUTDIR / "uv_visors_event_trades.csv", index=False)

display(event_summary.head(50))
display(event_summary[event_summary["robust_pass"]].head(50))

[   0.00s] event 1/35000: YELLOW_own_meanrev, w=500, th=1.25, h=250
[   5.77s] event 500/35000: ORANGE_own_meanrev, w=5000, th=2.0, h=2500
[  11.77s] event 1000/35000: MAGENTA_own_trend, w=5000, th=2.0, h=2500
[  17.44s] event 1500/35000: YELLOW_vs_AMBER_ratio_mr, w=5000, th=2.0, h=2500
[  24.12s] event 2000/35000: AMBER_resid_vs_YELLOW_poly3_mr_DISCOVERY, w=5000, th=2.0, h=2500
[  29.38s] event 2500/35000: YELLOW_vs_ORANGE_ratio_mr, w=5000, th=2.0, h=2500
[  35.93s] event 3000/35000: ORANGE_resid_vs_YELLOW_poly3_mr_DISCOVERY, w=5000, th=2.0, h=2500
[  41.93s] event 3500/35000: YELLOW_vs_RED_ratio_mr, w=5000, th=2.0, h=2500
[  48.83s] event 4000/35000: RED_resid_vs_YELLOW_poly3_mr_DISCOVERY, w=5000, th=2.0, h=2500
[  55.37s] event 4500/35000: YELLOW_vs_MAGENTA_ratio_mr, w=5000, th=2.0, h=2500
[  62.29s] event 5000/35000: MAGENTA_resid_vs_YELLOW_poly3_mr_DISCOVERY, w=5000, th=2.0, h=2500
[  68.55s] event 5500/35000: AMBER_vs_ORANGE_ratio_mr, w=5000, th=2.0, h=2500
[  75.35s] event 6000/

,candidate,family,signal_type,note,window,threshold,horizon,gross,q_trade,net_position,...,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,mean_hit_rate,min_hit_rate,pnl_per_event,one_day_dependency,robust_pass,robust_score
16008,lead_AMBER_to_ORANGE_follow_lag50,lead_lag,leader_return_follow,ORANGE follows AMBER return over lag 50,500,1.50,1500,60,"[0, 0, -60, 0, 0]",-60.0,...,310800.0,103600.0,64080.0,145500.0,0.833333,0.666667,17266.666667,0.468147,True,443126.666667
17924,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,1000,1.25,2500,60,"[0, 0, 0, 60, 0]",60.0,...,260100.0,86700.0,76020.0,104100.0,0.888889,0.666667,28900.000000,0.400231,True,416584.444444
15333,lead_AMBER_to_YELLOW_inverse_lag100,lead_lag,leader_return_inverse,YELLOW inversely follows AMBER return over lag...,1000,1.75,1500,60,"[60, 0, 0, 0, 0]",60.0,...,252780.0,84260.0,75900.0,93780.0,0.733333,0.600000,16852.000000,0.370995,True,408246.666667
19231,lead_ORANGE_to_YELLOW_follow_lag100,lead_lag,leader_return_follow,YELLOW follows ORANGE return over lag 100,1000,1.75,500,60,"[-60, 0, 0, 0, 0]",-60.0,...,259140.0,86380.0,70920.0,100080.0,0.725275,0.692308,6478.500000,0.386201,True,404606.373626
16203,lead_AMBER_to_ORANGE_follow_lag100,lead_lag,leader_return_follow,ORANGE follows AMBER return over lag 100,500,1.25,1500,60,"[0, 0, -60, 0, 0]",-60.0,...,267540.0,89180.0,65520.0,132660.0,0.777778,0.666667,14863.333333,0.495851,True,402468.888889
17909,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,500,1.50,2500,60,"[0, 0, 0, 60, 0]",60.0,...,254040.0,84680.0,68760.0,107160.0,0.888889,0.666667,28226.666667,0.421823,True,396004.444444
18332,lead_AMBER_to_MAGENTA_inverse_lag100,lead_lag,leader_return_inverse,MAGENTA inversely follows AMBER return over la...,1000,1.75,1000,60,"[0, 0, 0, 0, 60]",60.0,...,250020.0,83340.0,71100.0,107220.0,0.619048,0.571429,11905.714286,0.428846,True,395315.238095
13802,lead_YELLOW_to_RED_follow_lag1000,lead_lag,leader_return_follow,RED follows YELLOW return over lag 1000,500,1.25,1000,60,"[0, 0, 0, -60, 0]",-60.0,...,305040.0,101680.0,42300.0,132360.0,0.750000,0.625000,12710.000000,0.433910,True,393390.000000
23518,lead_RED_to_YELLOW_inverse_lag250,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 250,500,2.00,1500,60,"[60, 0, 0, 0, 0]",60.0,...,257520.0,85840.0,63000.0,125220.0,0.800000,0.800000,17168.000000,0.486253,True,387520.000000
23736,lead_RED_to_YELLOW_inverse_lag500,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 500,1000,2.00,500,60,"[60, 0, 0, 0, 0]",60.0,...,231180.0,77060.0,67680.0,84000.0,0.716667,0.700000,7224.375000,0.363353,True,370123.333333


,candidate,family,signal_type,note,window,threshold,horizon,gross,q_trade,net_position,...,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,mean_hit_rate,min_hit_rate,pnl_per_event,one_day_dependency,robust_pass,robust_score
16008,lead_AMBER_to_ORANGE_follow_lag50,lead_lag,leader_return_follow,ORANGE follows AMBER return over lag 50,500,1.50,1500,60,"[0, 0, -60, 0, 0]",-60.0,...,310800.0,103600.0,64080.0,145500.0,0.833333,0.666667,17266.666667,0.468147,True,443126.666667
17924,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,1000,1.25,2500,60,"[0, 0, 0, 60, 0]",60.0,...,260100.0,86700.0,76020.0,104100.0,0.888889,0.666667,28900.000000,0.400231,True,416584.444444
15333,lead_AMBER_to_YELLOW_inverse_lag100,lead_lag,leader_return_inverse,YELLOW inversely follows AMBER return over lag...,1000,1.75,1500,60,"[60, 0, 0, 0, 0]",60.0,...,252780.0,84260.0,75900.0,93780.0,0.733333,0.600000,16852.000000,0.370995,True,408246.666667
19231,lead_ORANGE_to_YELLOW_follow_lag100,lead_lag,leader_return_follow,YELLOW follows ORANGE return over lag 100,1000,1.75,500,60,"[-60, 0, 0, 0, 0]",-60.0,...,259140.0,86380.0,70920.0,100080.0,0.725275,0.692308,6478.500000,0.386201,True,404606.373626
16203,lead_AMBER_to_ORANGE_follow_lag100,lead_lag,leader_return_follow,ORANGE follows AMBER return over lag 100,500,1.25,1500,60,"[0, 0, -60, 0, 0]",-60.0,...,267540.0,89180.0,65520.0,132660.0,0.777778,0.666667,14863.333333,0.495851,True,402468.888889
17909,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,500,1.50,2500,60,"[0, 0, 0, 60, 0]",60.0,...,254040.0,84680.0,68760.0,107160.0,0.888889,0.666667,28226.666667,0.421823,True,396004.444444
18332,lead_AMBER_to_MAGENTA_inverse_lag100,lead_lag,leader_return_inverse,MAGENTA inversely follows AMBER return over la...,1000,1.75,1000,60,"[0, 0, 0, 0, 60]",60.0,...,250020.0,83340.0,71100.0,107220.0,0.619048,0.571429,11905.714286,0.428846,True,395315.238095
13802,lead_YELLOW_to_RED_follow_lag1000,lead_lag,leader_return_follow,RED follows YELLOW return over lag 1000,500,1.25,1000,60,"[0, 0, 0, -60, 0]",-60.0,...,305040.0,101680.0,42300.0,132360.0,0.750000,0.625000,12710.000000,0.433910,True,393390.000000
23518,lead_RED_to_YELLOW_inverse_lag250,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 250,500,2.00,1500,60,"[60, 0, 0, 0, 0]",60.0,...,257520.0,85840.0,63000.0,125220.0,0.800000,0.800000,17168.000000,0.486253,True,387520.000000
23736,lead_RED_to_YELLOW_inverse_lag500,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 500,1000,2.00,500,60,"[60, 0, 0, 0, 0]",60.0,...,231180.0,77060.0,67680.0,84000.0,0.716667,0.700000,7224.375000,0.363353,True,370123.333333


In [9]:
# =========================
# 6) Strict-position backtest on top event candidates
# This is the important validation stage.
# =========================

def strict_backtest_one_config(cand, window, entry_z, exit_z, max_hold, gross):
    q_low = q_from_weights(cand["pos_low_weights"], gross=gross)

    day_rows = []
    all_trades = []

    for day, panel in panels.items():
        mid, bid, ask = panel["mid"], panel["bid"], panel["ask"]
        sig = cand["signal_func"](mid)
        z = rolling_z(sig, window)

        trades = []
        i = window

        while i < len(mid) - 1:
            zi = z.iloc[i]
            if not np.isfinite(zi):
                i += 1
                continue

            if zi <= -entry_z:
                pos = q_low
                side = 1
            elif zi >= entry_z:
                pos = -q_low
                side = -1
            else:
                i += 1
                continue

            entry_i = i
            j = i + 1

            while j < len(mid) - 1:
                zj = z.iloc[j]
                hold = j - entry_i

                exit_hit = np.isfinite(zj) and abs(zj) <= exit_z
                hold_hit = hold >= max_hold
                end_hit = j >= len(mid) - 2

                if exit_hit or hold_hit or end_hit:
                    break
                j += 1

            exit_i = j
            pnl = exec_pnl(pos, entry_i, exit_i, bid, ask)

            trade = {
                "candidate": cand["candidate"],
                "family": cand["family"],
                "signal_type": cand["signal_type"],
                "note": cand["note"],
                "day": day,
                "window": window,
                "entry_z": entry_z,
                "exit_z": exit_z,
                "max_hold": max_hold,
                "gross": gross,
                "entry_idx": entry_i,
                "exit_idx": exit_i,
                "entry_ts": mid.index[entry_i],
                "exit_ts": mid.index[exit_i],
                "side": side,
                "entry_z_seen": float(z.iloc[entry_i]),
                "exit_z_seen": float(z.iloc[exit_i]) if np.isfinite(z.iloc[exit_i]) else np.nan,
                "hold": int(exit_i - entry_i),
                "exec_pnl": pnl,
                "q_trade": q_low.tolist(),
                "position": pos.tolist(),
            }
            trades.append(trade)
            all_trades.append(trade)

            # One strict open position at a time.
            i = exit_i + 1

        s = summarise_day_rows(trades)
        s.update({
            "candidate": cand["candidate"],
            "family": cand["family"],
            "signal_type": cand["signal_type"],
            "note": cand["note"],
            "day": day,
            "window": window,
            "entry_z": entry_z,
            "exit_z": exit_z,
            "max_hold": max_hold,
            "gross": gross,
            "q_trade": q_low.tolist(),
            "net_position": float(np.sum(q_low)),
            "gross_position": float(np.sum(np.abs(q_low))),
        })
        day_rows.append(s)

    day_df = pd.DataFrame(day_rows)
    total_pnl = day_df["day_pnl"].sum()
    total_trades = day_df["trade_count"].sum()
    active_days = int((day_df["trade_count"] > 0).sum())
    mean_hit = day_df["hit_rate"].mean()
    min_hit = day_df["hit_rate"].min()
    min_day = day_df["day_pnl"].min()
    max_day = day_df["day_pnl"].max()
    one_day_dep = abs(max_day) / abs(total_pnl) if total_pnl != 0 else np.inf

    summary = {
        "candidate": cand["candidate"],
        "family": cand["family"],
        "signal_type": cand["signal_type"],
        "note": cand["note"],
        "window": window,
        "entry_z": entry_z,
        "exit_z": exit_z,
        "max_hold": max_hold,
        "gross": gross,
        "q_trade": q_low.tolist(),
        "net_position": float(np.sum(q_low)),
        "gross_position": float(np.sum(np.abs(q_low))),
        "days": len(DAYS),
        "active_days": active_days,
        "total_trades": int(total_trades),
        "total_pnl": float(total_pnl),
        "mean_day_pnl": float(day_df["day_pnl"].mean()),
        "min_day_pnl": float(min_day),
        "max_day_pnl": float(max_day),
        "mean_hit_rate": float(mean_hit),
        "min_hit_rate": float(min_hit),
        "pnl_per_trade": float(total_pnl / total_trades) if total_trades else np.nan,
        "one_day_dependency": float(one_day_dep),
    }

    summary["robust_pass"] = (
        active_days == len(DAYS)
        and total_trades >= 5
        and min_day > 0
        and mean_hit > 0.55
    )
    summary["robust_score"] = robust_score(total_pnl, min_day, mean_hit, one_day_dep)

    return summary, day_df, all_trades


# Select candidates for strict testing:
# 1) top robust event candidates
# 2) top overall event candidates
# 3) manually force in suspected RED lead-lag and colour-mixing candidates
top_event_names = list(event_summary.head(80)["candidate"].unique())
top_robust_names = list(event_summary[event_summary["robust_pass"]].head(80)["candidate"].unique())

manual_names = [
    name for name in [c["candidate"] for c in candidates]
    if (
        "lead_RED_to_YELLOW" in name
        or "lead_RED_to_ORANGE" in name
        or "lead_RED_to_AMBER" in name
        or "lead_MAGENTA_to_RED" in name
        or "mix_" in name
        or "curvature_" in name
        or "red_extreme" in name.lower()
        or "spectrum_gradient" in name
    )
]

strict_names = list(dict.fromkeys(top_robust_names + top_event_names + manual_names))
strict_name_set = set(strict_names)
strict_candidates = [c for c in candidates if c["candidate"] in strict_name_set]

print("Strict candidates:", len(strict_candidates))


STRICT_WINDOWS = [1000, 2500, 4000, 5000]
STRICT_ENTRY_ZS = [1.25, 1.5, 1.75, 2.0]
STRICT_EXIT_ZS = [0.0, 0.25, 0.5]
STRICT_MAX_HOLDS = [1000, 1500, 2500]
STRICT_GROSSES = [50, 60, 72]

strict_configs = list(product(
    range(len(strict_candidates)),
    STRICT_WINDOWS,
    STRICT_ENTRY_ZS,
    STRICT_EXIT_ZS,
    STRICT_MAX_HOLDS,
    STRICT_GROSSES,
))

strict_summaries = []
strict_day_breakdowns = []
strict_trades = []

t0 = time.time()

for k, (ci, w, entry, exit_, hold, gross) in enumerate(strict_configs, start=1):
    cand = strict_candidates[ci]

    if k == 1 or k % 250 == 0 or k == len(strict_configs):
        print(
            f"[{time.time()-t0:7.2f}s] strict {k}/{len(strict_configs)}: "
            f"{cand['candidate']}, w={w}, entry={entry}, exit={exit_}, hold={hold}, gross={gross}"
        )

    summary, day_df, trades = strict_backtest_one_config(
        cand=cand,
        window=w,
        entry_z=entry,
        exit_z=exit_,
        max_hold=hold,
        gross=gross,
    )

    strict_summaries.append(summary)
    strict_day_breakdowns.append(day_df)
    strict_trades.extend(trades)

strict_summary = pd.DataFrame(strict_summaries).sort_values("robust_score", ascending=False)
strict_day_breakdown = pd.concat(strict_day_breakdowns, ignore_index=True)
strict_trades_df = pd.DataFrame(strict_trades)

strict_summary.to_csv(OUTDIR / "uv_visors_strict_summary.csv", index=False)
strict_summary[strict_summary["robust_pass"]].to_csv(OUTDIR / "uv_visors_strict_robust_only.csv", index=False)
strict_day_breakdown.to_csv(OUTDIR / "uv_visors_strict_day_breakdown.csv", index=False)
strict_trades_df.to_csv(OUTDIR / "uv_visors_strict_trades.csv", index=False)

display(strict_summary.head(50))
display(strict_summary[strict_summary["robust_pass"]].head(50))

Strict candidates: 100
[   0.00s] strict 1/43200: YELLOW_vs_ORANGE_spread_trend, w=1000, entry=1.25, exit=0.0, hold=1000, gross=50
[  13.21s] strict 250/43200: YELLOW_vs_ORANGE_spread_trend, w=4000, entry=1.5, exit=0.0, hold=2500, gross=50
[  23.75s] strict 500/43200: YELLOW_vs_ORANGE_logratio_trend, w=1000, entry=1.75, exit=0.25, hold=1500, gross=60
[  35.31s] strict 750/43200: YELLOW_vs_ORANGE_logratio_trend, w=4000, entry=2.0, exit=0.5, hold=1000, gross=72
[  46.72s] strict 1000/43200: YELLOW_vs_ORANGE_ratio_trend, w=2500, entry=1.5, exit=0.0, hold=1000, gross=50
[  56.87s] strict 1250/43200: YELLOW_vs_ORANGE_ratio_trend, w=5000, entry=1.75, exit=0.0, hold=2500, gross=60


KeyboardInterrupt: 

In [11]:
# =========================
# 6-FAST) Optimised strict-position backtest for UV VISORS
# Replacement for slow cell #6
# Uses event_summary + candidates + panels from previous cells
# =========================

import numpy as np
import pandas as pd
from pathlib import Path
import time
from itertools import product

FAST_OUTDIR = Path("analysis_outputs/uv_visors_fast_strict")
FAST_OUTDIR.mkdir(parents=True, exist_ok=True)

# Load from disk if needed
if "event_summary" not in globals():
    event_summary = pd.read_csv("analysis_outputs/uv_visors/uv_visors_event_summary.csv")

if "leadlag_diag" not in globals():
    try:
        leadlag_diag = pd.read_csv("analysis_outputs/uv_visors/uv_visors_leadlag_diagnostics.csv")
    except FileNotFoundError:
        leadlag_diag = None

candidate_by_name = {c["candidate"]: c for c in candidates}

# Do not allow leaky static polynomial discovery candidates into final strict test.
EXCLUDE_DISCOVERY = True

ev = event_summary.copy()
if EXCLUDE_DISCOVERY:
    ev = ev[~ev["candidate"].str.contains("DISCOVERY", na=False)].copy()

ev = ev.sort_values("robust_score", ascending=False).reset_index(drop=True)

# Robust pass may load as bool or string
if ev["robust_pass"].dtype == object:
    ev["robust_pass_bool"] = ev["robust_pass"].astype(str).str.lower().eq("true")
else:
    ev["robust_pass_bool"] = ev["robust_pass"].astype(bool)


# -------------------------
# Candidate selection
# -------------------------

selected_names = []

def add_names(names):
    for n in names:
        if n in candidate_by_name and n not in selected_names:
            selected_names.append(n)

# 1) top robust and top overall event candidates
add_names(ev[ev["robust_pass_bool"]].head(80)["candidate"].tolist())
add_names(ev.head(80)["candidate"].tolist())

# 2) family rescue: keep mixing/curvature/pairwise alive even if lead-lag dominates
for fam, n in [
    ("lead_lag", 60),
    ("pairwise", 25),
    ("colour_mixing", 20),
    ("colour_curvature", 10),
    ("colour_basket", 15),
    ("single", 10),
]:
    add_names(ev[ev["family"].eq(fam)].head(n)["candidate"].tolist())

# 3) lead-lag diagnostics rescue: map top diagnostic rows to exact candidate names
if leadlag_diag is not None:
    for _, r in leadlag_diag.head(40).iterrows():
        leader = COLOR_SHORT[r["leader"]]
        follower = COLOR_SHORT[r["follower"]]
        lag = int(r["lag"])
        mode = str(r["suggested_mode"])
        add_names([f"lead_{leader}_to_{follower}_{mode}_lag{lag}"])

# Limit for fast pass. Increase to 150 if still quick.
MAX_CANDIDATES_FAST = 120
selected_names = selected_names[:MAX_CANDIDATES_FAST]
selected_candidates = [candidate_by_name[n] for n in selected_names]

print("Selected strict candidates:", len(selected_candidates))
print(pd.Series([c["family"] for c in selected_candidates]).value_counts())


# -------------------------
# Fast caches
# -------------------------

panel_np = {}
for day, panel in panels.items():
    mid = panel["mid"]
    panel_np[day] = {
        "mid": mid,
        "bid_arr": panel["bid"].values.astype(float),
        "ask_arr": panel["ask"].values.astype(float),
        "ts_arr": mid.index.to_numpy(),
        "n": len(mid),
    }

def rolling_z_np(values, window):
    s = pd.Series(values).replace([np.inf, -np.inf], np.nan)
    mu = s.rolling(window, min_periods=window).mean()
    sd = s.rolling(window, min_periods=window).std(ddof=0)
    z = ((s - mu) / sd).replace([np.inf, -np.inf], np.nan)
    return z.to_numpy(dtype=float)

signal_cache = {}
z_cache = {}

# We only use windows that appear in the generated config list below,
# but define broad enough here.
STRICT_WINDOWS_FAST = [500, 1000, 1500, 2500, 4000, 5000]

print("Precomputing signals and z-scores...")
t0 = time.time()

for ci, cand in enumerate(selected_candidates, start=1):
    cname = cand["candidate"]

    if ci == 1 or ci % 20 == 0 or ci == len(selected_candidates):
        print(f"[{time.time()-t0:7.2f}s] candidate {ci}/{len(selected_candidates)}: {cname}")

    for day, pnp in panel_np.items():
        mid = pnp["mid"]
        sig = cand["signal_func"](mid)
        sig_arr = pd.Series(sig).to_numpy(dtype=float)
        signal_cache[(cname, day)] = sig_arr

        for w in STRICT_WINDOWS_FAST:
            z_cache[(cname, day, w)] = rolling_z_np(sig_arr, w)

print(f"Cache done in {time.time()-t0:.2f}s")


# -------------------------
# Fast strict execution
# -------------------------

def exec_pnl_np(pos, entry_i, exit_i, bid_arr, ask_arr):
    pos = np.asarray(pos, dtype=float)

    entry_bid = bid_arr[entry_i]
    entry_ask = ask_arr[entry_i]
    exit_bid = bid_arr[exit_i]
    exit_ask = ask_arr[exit_i]

    long = pos > 0
    short = pos < 0

    pnl = 0.0
    pnl += np.sum(pos[long] * (exit_bid[long] - entry_ask[long]))
    pnl += np.sum((-pos[short]) * (entry_bid[short] - exit_ask[short]))
    return float(pnl)


def strict_backtest_fast(cand, window, entry_z, exit_z, max_hold, gross):
    cname = cand["candidate"]
    q_low = q_from_weights(cand["pos_low_weights"], gross=gross)

    day_rows = []
    all_trades = []

    for day, pnp in panel_np.items():
        z = z_cache[(cname, day, window)]
        bid_arr = pnp["bid_arr"]
        ask_arr = pnp["ask_arr"]
        ts_arr = pnp["ts_arr"]
        n = pnp["n"]

        trades = []
        i = window

        while i < n - 2:
            zi = z[i]

            if not np.isfinite(zi):
                i += 1
                continue

            if zi <= -entry_z:
                pos = q_low
                side = 1
            elif zi >= entry_z:
                pos = -q_low
                side = -1
            else:
                i += 1
                continue

            entry_i = i
            max_j = min(n - 2, entry_i + int(max_hold))

            # First zero/exit-zone touch, otherwise max_hold/end.
            z_slice = z[entry_i + 1 : max_j + 1]
            exit_hits = np.where(np.isfinite(z_slice) & (np.abs(z_slice) <= exit_z))[0]

            if len(exit_hits) > 0:
                exit_i = entry_i + 1 + int(exit_hits[0])
            else:
                exit_i = max_j

            pnl = exec_pnl_np(pos, entry_i, exit_i, bid_arr, ask_arr)

            trade = {
                "candidate": cname,
                "family": cand["family"],
                "signal_type": cand["signal_type"],
                "note": cand["note"],
                "day": day,
                "window": window,
                "entry_z": entry_z,
                "exit_z": exit_z,
                "max_hold": max_hold,
                "gross": gross,
                "entry_idx": entry_i,
                "exit_idx": exit_i,
                "entry_ts": ts_arr[entry_i],
                "exit_ts": ts_arr[exit_i],
                "side": side,
                "entry_z_seen": float(z[entry_i]),
                "exit_z_seen": float(z[exit_i]) if np.isfinite(z[exit_i]) else np.nan,
                "hold": int(exit_i - entry_i),
                "exec_pnl": pnl,
                "q_trade": q_low.tolist(),
                "position": pos.tolist(),
            }

            trades.append(trade)
            all_trades.append(trade)

            # Strict: only one open position at a time.
            i = exit_i + 1

        if trades:
            pnls = np.array([t["exec_pnl"] for t in trades], dtype=float)
            trade_count = len(pnls)
            day_pnl = float(pnls.sum())
            hit_rate = float(np.mean(pnls > 0))
            avg_trade_pnl = float(np.mean(pnls))
            median_trade_pnl = float(np.median(pnls))
        else:
            trade_count = 0
            day_pnl = 0.0
            hit_rate = np.nan
            avg_trade_pnl = np.nan
            median_trade_pnl = np.nan

        day_rows.append({
            "candidate": cname,
            "family": cand["family"],
            "signal_type": cand["signal_type"],
            "note": cand["note"],
            "day": day,
            "window": window,
            "entry_z": entry_z,
            "exit_z": exit_z,
            "max_hold": max_hold,
            "gross": gross,
            "q_trade": q_low.tolist(),
            "net_position": float(np.sum(q_low)),
            "gross_position": float(np.sum(np.abs(q_low))),
            "trade_count": trade_count,
            "day_pnl": day_pnl,
            "hit_rate": hit_rate,
            "avg_trade_pnl": avg_trade_pnl,
            "median_trade_pnl": median_trade_pnl,
        })

    day_df = pd.DataFrame(day_rows)

    total_pnl = float(day_df["day_pnl"].sum())
    total_trades = int(day_df["trade_count"].sum())
    active_days = int((day_df["trade_count"] > 0).sum())

    mean_hit = float(day_df["hit_rate"].mean())
    min_hit = float(day_df["hit_rate"].min())
    min_day = float(day_df["day_pnl"].min())
    max_day = float(day_df["day_pnl"].max())
    one_day_dep = abs(max_day) / abs(total_pnl) if total_pnl != 0 else np.inf

    summary = {
        "candidate": cname,
        "family": cand["family"],
        "signal_type": cand["signal_type"],
        "note": cand["note"],
        "window": window,
        "entry_z": entry_z,
        "exit_z": exit_z,
        "max_hold": max_hold,
        "gross": gross,
        "q_trade": q_low.tolist(),
        "net_position": float(np.sum(q_low)),
        "gross_position": float(np.sum(np.abs(q_low))),
        "days": len(DAYS),
        "active_days": active_days,
        "total_trades": total_trades,
        "total_pnl": total_pnl,
        "mean_day_pnl": float(day_df["day_pnl"].mean()),
        "min_day_pnl": min_day,
        "max_day_pnl": max_day,
        "mean_hit_rate": mean_hit,
        "min_hit_rate": min_hit,
        "pnl_per_trade": float(total_pnl / total_trades) if total_trades else np.nan,
        "one_day_dependency": float(one_day_dep),
    }

    summary["robust_pass"] = (
        active_days == len(DAYS)
        and total_trades >= 5
        and min_day > 0
        and mean_hit > 0.55
    )
    summary["robust_score"] = robust_score(total_pnl, min_day, mean_hit, one_day_dep)

    return summary, day_df, all_trades

Selected strict candidates: 87
lead_lag            57
colour_basket        8
colour_mixing        7
single               6
pairwise             5
colour_curvature     4
Name: count, dtype: int64
Precomputing signals and z-scores...
[   0.00s] candidate 1/87: lead_AMBER_to_ORANGE_follow_lag50
[   0.11s] candidate 20/87: lead_AMBER_to_MAGENTA_follow_lag1000
[   0.25s] candidate 40/87: RED_vs_MAGENTA_logratio_trend
[   0.38s] candidate 60/87: basket_magenta_extreme_vs_rest_mr
[   0.52s] candidate 80/87: lead_ORANGE_to_YELLOW_inverse_lag500
[   0.58s] candidate 87/87: lead_ORANGE_to_RED_follow_lag1000
Cache done in 0.59s


In [12]:
# =========================
# 6B-FAST) Build targeted strict configs and run
# =========================

def neighbours(value, allowed):
    allowed = sorted(set(allowed))
    value = int(value)
    if value not in allowed:
        allowed.append(value)
        allowed = sorted(set(allowed))

    i = allowed.index(value)
    out = {value}
    if i > 0:
        out.add(allowed[i - 1])
    if i < len(allowed) - 1:
        out.add(allowed[i + 1])
    return sorted(out)


ALLOWED_WINDOWS = [500, 1000, 1500, 2500, 4000, 5000]
ALLOWED_ENTRIES = [1.25, 1.5, 1.75, 2.0, 2.25]
ALLOWED_EXITS = [0.0, 0.25, 0.5]
ALLOWED_HOLDS = [500, 1000, 1500, 2500, 4000]
ALLOWED_GROSSES = [40, 60, 72, 80]

# Use best event rows per selected candidate to form strict configs.
selected_event_rows = (
    ev[ev["candidate"].isin(selected_names)]
    .sort_values("robust_score", ascending=False)
    .groupby("candidate")
    .head(5)
    .copy()
)

configs = set()

for _, r in selected_event_rows.iterrows():
    cname = r["candidate"]

    base_w = int(r["window"])
    base_entry = float(r["threshold"])
    base_hold = int(r["horizon"])

    ws = neighbours(base_w, ALLOWED_WINDOWS)
    entries = sorted(set([base_entry] + neighbours(round(base_entry * 4) / 4, ALLOWED_ENTRIES)))
    entries = [e for e in entries if e in ALLOWED_ENTRIES]

    holds = neighbours(base_hold, ALLOWED_HOLDS)

    for w, entry_z, exit_z, hold, gross in product(
        ws,
        entries,
        ALLOWED_EXITS,
        holds,
        ALLOWED_GROSSES,
    ):
        configs.add((cname, int(w), float(entry_z), float(exit_z), int(hold), int(gross)))

# Add a small canonical grid for the highest-confidence names.
elite_names = ev.head(25)["candidate"].tolist()
for cname in elite_names:
    if cname not in selected_names:
        continue

    for w, entry_z, exit_z, hold, gross in product(
        [500, 1000, 2500],
        [1.25, 1.5, 1.75, 2.0],
        [0.0, 0.25, 0.5],
        [1000, 1500, 2500],
        [60, 72, 80],
    ):
        configs.add((cname, w, entry_z, exit_z, hold, gross))

configs = sorted(configs)
print("Strict configs:", len(configs))

strict_summaries = []
strict_day_breakdowns = []
strict_trades = []

t0 = time.time()

for k, (cname, w, entry_z, exit_z, hold, gross) in enumerate(configs, start=1):
    cand = candidate_by_name[cname]

    if k == 1 or k % 500 == 0 or k == len(configs):
        print(
            f"[{time.time()-t0:7.2f}s] strict {k}/{len(configs)}: "
            f"{cname}, w={w}, entry={entry_z}, exit={exit_z}, hold={hold}, gross={gross}"
        )

    summary, day_df, trades = strict_backtest_fast(
        cand=cand,
        window=w,
        entry_z=entry_z,
        exit_z=exit_z,
        max_hold=hold,
        gross=gross,
    )

    strict_summaries.append(summary)
    strict_day_breakdowns.append(day_df)
    strict_trades.extend(trades)

strict_summary_fast = pd.DataFrame(strict_summaries).sort_values("robust_score", ascending=False)
strict_day_breakdown_fast = pd.concat(strict_day_breakdowns, ignore_index=True)
strict_trades_fast = pd.DataFrame(strict_trades)

strict_summary_fast.to_csv(FAST_OUTDIR / "uv_visors_strict_summary_fast.csv", index=False)
strict_summary_fast[strict_summary_fast["robust_pass"]].to_csv(FAST_OUTDIR / "uv_visors_strict_robust_only_fast.csv", index=False)
strict_day_breakdown_fast.to_csv(FAST_OUTDIR / "uv_visors_strict_day_breakdown_fast.csv", index=False)
strict_trades_fast.to_csv(FAST_OUTDIR / "uv_visors_strict_trades_fast.csv", index=False)

print("Top strict summary:")
display(strict_summary_fast.head(50))

print("Robust only:")
display(strict_summary_fast[strict_summary_fast["robust_pass"]].head(50))

Strict configs: 54780
[   0.00s] strict 1/54780: AMBER_own_meanrev, w=500, entry=1.25, exit=0.0, hold=500, gross=40
[   2.27s] strict 500/54780: AMBER_own_meanrev, w=2500, entry=1.25, exit=0.5, hold=1000, gross=80
[   4.36s] strict 1000/54780: AMBER_vs_MAGENTA_ratio_mr, w=1500, entry=2.0, exit=0.0, hold=500, gross=80
[   6.62s] strict 1500/54780: MAGENTA_own_meanrev, w=1500, entry=1.75, exit=0.25, hold=2500, gross=80
[   8.50s] strict 2000/54780: MAGENTA_own_meanrev, w=5000, entry=1.25, exit=0.25, hold=1000, gross=80
[  11.19s] strict 2500/54780: ORANGE_own_meanrev, w=1000, entry=1.5, exit=0.25, hold=1000, gross=80
[  12.95s] strict 3000/54780: RED_vs_MAGENTA_logratio_trend, w=500, entry=1.25, exit=0.25, hold=2500, gross=80
[  15.14s] strict 3500/54780: RED_vs_MAGENTA_logratio_trend, w=1500, entry=1.75, exit=0.25, hold=1500, gross=80
[  17.35s] strict 4000/54780: YELLOW_own_meanrev, w=1000, entry=2.0, exit=0.25, hold=1500, gross=80
[  19.60s] strict 4500/54780: YELLOW_vs_ORANGE_lograti

,candidate,family,signal_type,note,window,entry_z,exit_z,max_hold,gross,q_trade,...,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
20968,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,500,1.50,0.00,2500,80,"[0, 0, 0, 80, 0]",...,378720.0,126240.000000,116000.0,136560.0,0.833333,0.750000,31560.000000,0.360583,True,614886.666667
42547,lead_RED_to_YELLOW_inverse_lag250,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 250,500,2.00,0.00,1500,80,"[80, 0, 0, 0, 0]",...,405040.0,135013.333333,96480.0,177920.0,0.833333,0.833333,22502.222222,0.439265,True,602166.666667
21090,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,1000,1.25,0.00,2500,80,"[0, 0, 0, 80, 0]",...,363120.0,121040.000000,106640.0,136560.0,0.805556,0.666667,33010.909091,0.376074,True,580427.777778
20967,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,500,1.50,0.00,2500,72,"[0, 0, 0, 72, 0]",...,340848.0,113616.000000,104400.0,122904.0,0.833333,0.750000,28404.000000,0.360583,True,553814.666667
17749,lead_AMBER_to_ORANGE_follow_lag50,lead_lag,leader_return_follow,ORANGE follows AMBER return over lag 50,500,1.50,0.00,1500,80,"[0, 0, -80, 0, 0]",...,387520.0,129173.333333,78320.0,175680.0,0.714286,0.571429,18453.333333,0.453344,True,547731.428571
41977,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 1000,500,1.75,0.00,1500,80,"[80, 0, 0, 0, 0]",...,335040.0,111680.000000,102320.0,126320.0,0.766667,0.666667,19708.235294,0.377030,True,543513.333333
42546,lead_RED_to_YELLOW_inverse_lag250,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 250,500,2.00,0.00,1500,72,"[72, 0, 0, 0, 0]",...,364536.0,121512.000000,86832.0,160128.0,0.833333,0.833333,20252.000000,0.439265,True,542366.666667
48313,lead_YELLOW_to_RED_follow_lag1000,lead_lag,leader_return_follow,RED follows YELLOW return over lag 1000,500,1.25,0.00,1000,80,"[0, 0, 0, -80, 0]",...,418480.0,139493.333333,57760.0,182480.0,0.777778,0.666667,15499.259259,0.436054,True,537888.888889
42013,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 1000,500,2.00,0.00,1500,80,"[80, 0, 0, 0, 0]",...,340240.0,113413.333333,92320.0,134320.0,0.711111,0.666667,20014.117647,0.394780,True,528435.555556
21089,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,1000,1.25,0.00,2500,72,"[0, 0, 0, 72, 0]",...,326808.0,108936.000000,95976.0,122904.0,0.805556,0.666667,29709.818182,0.376074,True,522787.777778


Robust only:


,candidate,family,signal_type,note,window,entry_z,exit_z,max_hold,gross,q_trade,...,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
20968,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,500,1.50,0.00,2500,80,"[0, 0, 0, 80, 0]",...,378720.0,126240.000000,116000.0,136560.0,0.833333,0.750000,31560.000000,0.360583,True,614886.666667
42547,lead_RED_to_YELLOW_inverse_lag250,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 250,500,2.00,0.00,1500,80,"[80, 0, 0, 0, 0]",...,405040.0,135013.333333,96480.0,177920.0,0.833333,0.833333,22502.222222,0.439265,True,602166.666667
21090,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,1000,1.25,0.00,2500,80,"[0, 0, 0, 80, 0]",...,363120.0,121040.000000,106640.0,136560.0,0.805556,0.666667,33010.909091,0.376074,True,580427.777778
20967,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,500,1.50,0.00,2500,72,"[0, 0, 0, 72, 0]",...,340848.0,113616.000000,104400.0,122904.0,0.833333,0.750000,28404.000000,0.360583,True,553814.666667
17749,lead_AMBER_to_ORANGE_follow_lag50,lead_lag,leader_return_follow,ORANGE follows AMBER return over lag 50,500,1.50,0.00,1500,80,"[0, 0, -80, 0, 0]",...,387520.0,129173.333333,78320.0,175680.0,0.714286,0.571429,18453.333333,0.453344,True,547731.428571
41977,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 1000,500,1.75,0.00,1500,80,"[80, 0, 0, 0, 0]",...,335040.0,111680.000000,102320.0,126320.0,0.766667,0.666667,19708.235294,0.377030,True,543513.333333
42546,lead_RED_to_YELLOW_inverse_lag250,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 250,500,2.00,0.00,1500,72,"[72, 0, 0, 0, 0]",...,364536.0,121512.000000,86832.0,160128.0,0.833333,0.833333,20252.000000,0.439265,True,542366.666667
48313,lead_YELLOW_to_RED_follow_lag1000,lead_lag,leader_return_follow,RED follows YELLOW return over lag 1000,500,1.25,0.00,1000,80,"[0, 0, 0, -80, 0]",...,418480.0,139493.333333,57760.0,182480.0,0.777778,0.666667,15499.259259,0.436054,True,537888.888889
42013,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,leader_return_inverse,YELLOW inversely follows RED return over lag 1000,500,2.00,0.00,1500,80,"[80, 0, 0, 0, 0]",...,340240.0,113413.333333,92320.0,134320.0,0.711111,0.666667,20014.117647,0.394780,True,528435.555556
21089,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,1000,1.25,0.00,2500,72,"[0, 0, 0, 72, 0]",...,326808.0,108936.000000,95976.0,122904.0,0.805556,0.666667,29709.818182,0.376074,True,522787.777778


In [13]:
# =========================
# 7-FAST) Best strict config breakdown
# =========================

best = strict_summary_fast.iloc[0]

print("BEST STRICT CONFIG")
display(best.to_frame().T)

best_day_fast = strict_day_breakdown_fast[
    (strict_day_breakdown_fast["candidate"] == best["candidate"])
    & (strict_day_breakdown_fast["window"] == best["window"])
    & (strict_day_breakdown_fast["entry_z"] == best["entry_z"])
    & (strict_day_breakdown_fast["exit_z"] == best["exit_z"])
    & (strict_day_breakdown_fast["max_hold"] == best["max_hold"])
    & (strict_day_breakdown_fast["gross"] == best["gross"])
].copy()

best_trades_fast = strict_trades_fast[
    (strict_trades_fast["candidate"] == best["candidate"])
    & (strict_trades_fast["window"] == best["window"])
    & (strict_trades_fast["entry_z"] == best["entry_z"])
    & (strict_trades_fast["exit_z"] == best["exit_z"])
    & (strict_trades_fast["max_hold"] == best["max_hold"])
    & (strict_trades_fast["gross"] == best["gross"])
].copy()

print("BEST CONFIG DAY BREAKDOWN")
display(best_day_fast)

print("BEST CONFIG TRADES")
display(best_trades_fast)

best_day_fast.to_csv(FAST_OUTDIR / "uv_visors_best_day_breakdown_fast.csv", index=False)
best_trades_fast.to_csv(FAST_OUTDIR / "uv_visors_best_trades_fast.csv", index=False)

family_view_fast = (
    strict_summary_fast
    .groupby("family")
    .agg(
        best_total_pnl=("total_pnl", "max"),
        best_robust_score=("robust_score", "max"),
        robust_count=("robust_pass", "sum"),
        configs=("candidate", "count"),
    )
    .sort_values("best_robust_score", ascending=False)
)

print("FAMILY COMPARISON")
display(family_view_fast)

family_view_fast.to_csv(FAST_OUTDIR / "uv_visors_family_comparison_fast.csv")

BEST STRICT CONFIG


,candidate,family,signal_type,note,window,entry_z,exit_z,max_hold,gross,q_trade,...,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
20968,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,500,1.5,0.0,2500,80,"[0, 0, 0, 80, 0]",...,378720.0,126240.0,116000.0,136560.0,0.833333,0.75,31560.0,0.360583,True,614886.666667


BEST CONFIG DAY BREAKDOWN


,candidate,family,signal_type,note,day,window,entry_z,exit_z,max_hold,gross,q_trade,net_position,gross_position,trade_count,day_pnl,hit_rate,avg_trade_pnl,median_trade_pnl
62904,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.0,2500,80,"[0, 0, 0, 80, 0]",80.0,80.0,4,116000.0,0.75,29000.0,30840.0
62905,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,3,500,1.5,0.0,2500,80,"[0, 0, 0, 80, 0]",80.0,80.0,4,136560.0,0.75,34140.0,34680.0
62906,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,4,500,1.5,0.0,2500,80,"[0, 0, 0, 80, 0]",80.0,80.0,4,126160.0,1.00,31540.0,33240.0


BEST CONFIG TRADES


,candidate,family,signal_type,note,day,window,entry_z,exit_z,max_hold,gross,...,exit_idx,entry_ts,exit_ts,side,entry_z_seen,exit_z_seen,hold,exec_pnl,q_trade,position
992781,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.0,2500,80,...,4221,172100,422100,1,-1.509432,1.094972,2500,37360.0,"[0, 0, 0, 80, 0]","[0, 0, 0, 80, 0]"
992782,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.0,2500,80,...,6740,424000,674000,-1,1.505966,-2.123194,2500,56800.0,"[0, 0, 0, 80, 0]","[0, 0, 0, -80, 0]"
992783,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.0,2500,80,...,9241,674100,924100,1,-2.071663,-2.397480,2500,-2480.0,"[0, 0, 0, 80, 0]","[0, 0, 0, 80, 0]"
992784,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.0,2500,80,...,9998,924200,999800,1,-2.659828,1.199641,756,24320.0,"[0, 0, 0, 80, 0]","[0, 0, 0, 80, 0]"
992785,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,3,500,1.5,0.0,2500,80,...,4137,163700,413700,1,-1.575827,-1.057217,2500,22720.0,"[0, 0, 0, 80, 0]","[0, 0, 0, 80, 0]"
992786,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,3,500,1.5,0.0,2500,80,...,6732,423200,673200,-1,1.550373,1.271245,2500,46640.0,"[0, 0, 0, 80, 0]","[0, 0, 0, -80, 0]"
992787,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,3,500,1.5,0.0,2500,80,...,9455,695500,945500,1,-1.623993,1.749007,2500,73520.0,"[0, 0, 0, 80, 0]","[0, 0, 0, 80, 0]"
992788,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,3,500,1.5,0.0,2500,80,...,9998,945600,999800,-1,1.799073,-1.401258,542,-6320.0,"[0, 0, 0, 80, 0]","[0, 0, 0, -80, 0]"
992789,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,4,500,1.5,0.0,2500,80,...,3999,149900,399900,-1,2.057734,0.148917,2500,44480.0,"[0, 0, 0, 80, 0]","[0, 0, 0, -80, 0]"
992790,lead_AMBER_to_RED_inverse_lag1000,lead_lag,leader_return_inverse,RED inversely follows AMBER return over lag 1000,4,500,1.5,0.0,2500,80,...,6626,412600,662600,1,-1.525797,1.648370,2500,49040.0,"[0, 0, 0, 80, 0]","[0, 0, 0, 80, 0]"


FAMILY COMPARISON


,best_total_pnl,best_robust_score,robust_count,configs
family,,,,
lead_lag,418480.0,614886.666667,4303,37608
pairwise,285000.0,413708.888889,472,3108
single,307360.0,364760.000000,324,2640
colour_curvature,231240.0,342373.968254,148,2400
colour_mixing,231240.0,342373.968254,200,4188
colour_basket,180640.0,234397.619048,1718,4836


In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
import itertools
import time

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

UV_PRODUCTS = [
    "UV_VISOR_YELLOW",
    "UV_VISOR_AMBER",
    "UV_VISOR_ORANGE",
    "UV_VISOR_RED",
    "UV_VISOR_MAGENTA",
]

COLOUR_TO_PRODUCT = {
    "YELLOW": "UV_VISOR_YELLOW",
    "AMBER": "UV_VISOR_AMBER",
    "ORANGE": "UV_VISOR_ORANGE",
    "RED": "UV_VISOR_RED",
    "MAGENTA": "UV_VISOR_MAGENTA",
}

PRODUCT_TO_COLOUR = {v: k for k, v in COLOUR_TO_PRODUCT.items()}

OUT_DIR = Path("analysis_outputs/uv_visors_validation")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def auto_find_price_df():
    """
    Tries to find your existing price dataframe.
    Edit this manually if your dataframe has a different name.
    """
    candidate_names = [
        "prices_df",
        "price_df",
        "prices",
        "df_prices",
        "prices_train",
        "df",
    ]
    for name in candidate_names:
        if name in globals():
            obj = globals()[name]
            if isinstance(obj, pd.DataFrame):
                cols = set(obj.columns)
                if {"product", "timestamp"}.issubset(cols):
                    print(f"Using dataframe: {name}, shape={obj.shape}")
                    return obj.copy()
    raise NameError(
        "Could not auto-find a price dataframe. Set PRICE_DF manually, e.g. PRICE_DF = prices_df.copy()"
    )


def prepare_uv_mid_by_day(price_df: pd.DataFrame, products=UV_PRODUCTS):
    """
    Converts long-format IMC price dataframe into:
    uv_by_day[day] = dataframe indexed by timestamp, columns = UV products, values = mid prices.
    """
    df = price_df.copy()

    if "day" not in df.columns:
        raise ValueError("Expected a 'day' column in the price dataframe.")

    if "mid_price" not in df.columns:
        if {"bid_price_1", "ask_price_1"}.issubset(df.columns):
            df["mid_price"] = (df["bid_price_1"] + df["ask_price_1"]) / 2
        else:
            raise ValueError("Need either 'mid_price' or bid_price_1/ask_price_1 columns.")

    df = df[df["product"].isin(products)].copy()

    uv_by_day = {}
    for day, g in df.groupby("day"):
        pivot = (
            g.pivot_table(
                index="timestamp",
                columns="product",
                values="mid_price",
                aggfunc="last",
            )
            .sort_index()
            .reindex(columns=products)
        )

        # light forward/back fill in case there are tiny gaps
        pivot = pivot.ffill().bfill()

        if pivot[products].isna().any().any():
            print(f"Warning: day {day} still has NaNs after fill.")

        uv_by_day[int(day)] = pivot

    print("Prepared days:", sorted(uv_by_day.keys()))
    for day, mid in uv_by_day.items():
        print(f"Day {day}: shape={mid.shape}, products={list(mid.columns)}")

    return uv_by_day


# Use existing uv_by_day if already created, otherwise build it.
if "uv_by_day" not in globals():
    PRICE_DF = auto_find_price_df()
    uv_by_day = prepare_uv_mid_by_day(PRICE_DF)

Using dataframe: prices, shape=(1500000, 19)
Prepared days: [2, 3, 4]
Day 2: shape=(10000, 5), products=['UV_VISOR_YELLOW', 'UV_VISOR_AMBER', 'UV_VISOR_ORANGE', 'UV_VISOR_RED', 'UV_VISOR_MAGENTA']
Day 3: shape=(10000, 5), products=['UV_VISOR_YELLOW', 'UV_VISOR_AMBER', 'UV_VISOR_ORANGE', 'UV_VISOR_RED', 'UV_VISOR_MAGENTA']
Day 4: shape=(10000, 5), products=['UV_VISOR_YELLOW', 'UV_VISOR_AMBER', 'UV_VISOR_ORANGE', 'UV_VISOR_RED', 'UV_VISOR_MAGENTA']


In [15]:
def rolling_z(raw: pd.Series, window: int) -> pd.Series:
    mu = raw.rolling(window, min_periods=window).mean()
    sd = raw.rolling(window, min_periods=window).std(ddof=0)
    return (raw - mu) / sd.replace(0, np.nan)


def make_q_lead_lag(follower: str, relation: str, gross: int) -> np.ndarray:
    """
    q_base is the position taken when z <= -entry_z.

    relation='follow':
      leader down => follower expected down => short follower.
      q_base follower = -gross.

    relation='inverse':
      leader down => follower expected up => long follower.
      q_base follower = +gross.
    """
    q = np.zeros(len(UV_PRODUCTS), dtype=float)
    idx = UV_PRODUCTS.index(follower)

    if relation == "follow":
        q[idx] = -gross
    elif relation == "inverse":
        q[idx] = gross
    else:
        raise ValueError(f"Unknown relation: {relation}")

    return q


def make_q_mix(target: str, a: str, b: str, wa: float, gross: int) -> np.ndarray:
    """
    Mean-reversion blend q_base for z <= -entry_z:
      target cheap vs blend => long target, short blend.
    """
    q = np.zeros(len(UV_PRODUCTS), dtype=float)

    ia = UV_PRODUCTS.index(a)
    ib = UV_PRODUCTS.index(b)
    it = UV_PRODUCTS.index(target)

    qa = round(gross * wa)
    qb = gross - qa

    q[it] += gross
    q[ia] -= qa
    q[ib] -= qb

    return q


def compute_signal_z(mid_df: pd.DataFrame, spec: dict, window: int) -> np.ndarray:
    """
    Supports:
      kind='lead_lag'
      kind='mix_mr'
      kind='pair_ratio'
      kind='pair_spread'
    """
    kind = spec["kind"]

    if kind == "lead_lag":
        leader = spec["leader"]
        lag = int(spec["lag"])
        raw = mid_df[leader].diff(lag)

    elif kind == "mix_mr":
        target = spec["target"]
        a = spec["a"]
        b = spec["b"]
        wa = float(spec["wa"])
        blend = wa * mid_df[a] + (1 - wa) * mid_df[b]
        raw = mid_df[target] - blend

    elif kind == "pair_ratio":
        a = spec["a"]
        b = spec["b"]
        raw = np.log(mid_df[a] / mid_df[b])

    elif kind == "pair_spread":
        a = spec["a"]
        b = spec["b"]
        raw = mid_df[a] - mid_df[b]

    else:
        raise ValueError(f"Unknown spec kind: {kind}")

    return rolling_z(raw, window).to_numpy(dtype=float)


def q_for_spec(spec: dict, gross: int) -> np.ndarray:
    kind = spec["kind"]

    if kind == "lead_lag":
        return make_q_lead_lag(
            follower=spec["follower"],
            relation=spec["relation"],
            gross=gross,
        )

    if kind == "mix_mr":
        return make_q_mix(
            target=spec["target"],
            a=spec["a"],
            b=spec["b"],
            wa=spec["wa"],
            gross=gross,
        )

    if kind in {"pair_ratio", "pair_spread"}:
        # Mean-reversion pair default:
        # z <= -entry means a cheap vs b => long a, short b.
        q = np.zeros(len(UV_PRODUCTS), dtype=float)
        ia = UV_PRODUCTS.index(spec["a"])
        ib = UV_PRODUCTS.index(spec["b"])
        q[ia] = gross / 2
        q[ib] = -gross / 2
        return q

    raise ValueError(f"Unknown spec kind: {kind}")


def find_exit_idx(z, entry_i, max_j, entry_side, exit_mode, exit_z, entry_z):
    """
    entry_side:
      +1 means entered on negative z, position = q_base
      -1 means entered on positive z, position = -q_base
    """
    z_slice = z[entry_i + 1 : max_j + 1]

    if len(z_slice) == 0:
        return max_j

    finite = np.isfinite(z_slice)

    if exit_mode == "fixed_hold":
        return max_j

    if exit_mode == "zero_cross":
        if entry_side == 1:
            hits = np.where(finite & (z_slice >= 0))[0]
        else:
            hits = np.where(finite & (z_slice <= 0))[0]
        return entry_i + 1 + int(hits[0]) if len(hits) else max_j

    if exit_mode == "abs_band":
        hits = np.where(finite & (np.abs(z_slice) <= exit_z))[0]
        return entry_i + 1 + int(hits[0]) if len(hits) else max_j

    if exit_mode == "signal_flip":
        if entry_side == 1:
            hits = np.where(finite & (z_slice >= entry_z))[0]
        else:
            hits = np.where(finite & (z_slice <= -entry_z))[0]
        return entry_i + 1 + int(hits[0]) if len(hits) else max_j

    raise ValueError(f"Unknown exit_mode: {exit_mode}")


def strict_backtest_one_day(
    mid_df: pd.DataFrame,
    spec: dict,
    window: int,
    entry_z: float,
    exit_z: float,
    max_hold: int,
    gross: int,
    exit_mode: str,
    day: int,
):
    """
    One-position-at-a-time strict backtest.
    Uses mid-to-mid PnL.
    """
    prices = mid_df[UV_PRODUCTS].to_numpy(dtype=float)
    z = compute_signal_z(mid_df, spec, window)
    q_base = q_for_spec(spec, gross)

    n = len(mid_df)
    trades = []

    start_i = max(window + int(spec.get("lag", 0)) + 1, window + 1)
    i = start_i

    while i < n - 1:
        zi = z[i]

        if not np.isfinite(zi):
            i += 1
            continue

        if zi <= -entry_z:
            side = 1
        elif zi >= entry_z:
            side = -1
        else:
            i += 1
            continue

        max_j = min(n - 1, i + max_hold)

        exit_i = find_exit_idx(
            z=z,
            entry_i=i,
            max_j=max_j,
            entry_side=side,
            exit_mode=exit_mode,
            exit_z=exit_z,
            entry_z=entry_z,
        )

        pos = side * q_base
        pnl = float(np.dot(pos, prices[exit_i] - prices[i]))

        trades.append(
            {
                "candidate": spec["name"],
                "family": spec.get("family", spec["kind"]),
                "kind": spec["kind"],
                "note": spec.get("note", ""),
                "day": day,
                "window": window,
                "entry_z": entry_z,
                "exit_z": exit_z,
                "exit_mode": exit_mode,
                "max_hold": max_hold,
                "gross": gross,
                "q_trade": list(q_base),
                "net_position": float(q_base.sum()),
                "gross_position": float(np.abs(q_base).sum()),
                "entry_idx": int(i),
                "exit_idx": int(exit_i),
                "entry_ts": int(mid_df.index[i]),
                "exit_ts": int(mid_df.index[exit_i]),
                "side": int(side),
                "entry_z_seen": float(zi),
                "exit_z_seen": float(z[exit_i]) if np.isfinite(z[exit_i]) else np.nan,
                "hold": int(exit_i - i),
                "exec_pnl": pnl,
                "position": list(pos),
            }
        )

        i = exit_i + 1

    return trades


def summarize_trades(trades_df: pd.DataFrame, spec: dict, cfg: dict):
    if trades_df.empty:
        day_rows = []
        for day in sorted(uv_by_day.keys()):
            day_rows.append(
                {
                    "candidate": spec["name"],
                    "family": spec.get("family", spec["kind"]),
                    "kind": spec["kind"],
                    "note": spec.get("note", ""),
                    "day": day,
                    **cfg,
                    "trade_count": 0,
                    "day_pnl": 0.0,
                    "hit_rate": np.nan,
                    "avg_trade_pnl": np.nan,
                    "median_trade_pnl": np.nan,
                }
            )
        day_df = pd.DataFrame(day_rows)
    else:
        day_df = (
            trades_df.groupby("day")
            .agg(
                trade_count=("exec_pnl", "size"),
                day_pnl=("exec_pnl", "sum"),
                hit_rate=("exec_pnl", lambda x: float((x > 0).mean())),
                avg_trade_pnl=("exec_pnl", "mean"),
                median_trade_pnl=("exec_pnl", "median"),
            )
            .reset_index()
        )

        for day in sorted(uv_by_day.keys()):
            if day not in set(day_df["day"]):
                day_df = pd.concat(
                    [
                        day_df,
                        pd.DataFrame(
                            [
                                {
                                    "day": day,
                                    "trade_count": 0,
                                    "day_pnl": 0.0,
                                    "hit_rate": np.nan,
                                    "avg_trade_pnl": np.nan,
                                    "median_trade_pnl": np.nan,
                                }
                            ]
                        ),
                    ],
                    ignore_index=True,
                )

        day_df = day_df.sort_values("day").reset_index(drop=True)

        for k, v in cfg.items():
            day_df[k] = v

        day_df["candidate"] = spec["name"]
        day_df["family"] = spec.get("family", spec["kind"])
        day_df["kind"] = spec["kind"]
        day_df["note"] = spec.get("note", "")

    total_pnl = float(day_df["day_pnl"].sum())
    mean_day_pnl = float(day_df["day_pnl"].mean())
    min_day_pnl = float(day_df["day_pnl"].min())
    max_day_pnl = float(day_df["day_pnl"].max())
    total_trades = int(day_df["trade_count"].sum())

    hit_rates = day_df["hit_rate"].dropna()
    mean_hit_rate = float(hit_rates.mean()) if len(hit_rates) else np.nan
    min_hit_rate = float(hit_rates.min()) if len(hit_rates) else np.nan

    pnl_per_trade = total_pnl / total_trades if total_trades else np.nan
    one_day_dependency = max_day_pnl / total_pnl if total_pnl > 0 else np.inf

    robust_pass = (
        total_pnl > 0
        and min_day_pnl > 0
        and total_trades >= 3
        and one_day_dependency <= 0.60
    )

    robust_score = (
        total_pnl
        + 2.0 * min_day_pnl
        + 5000.0 * (0 if np.isnan(min_hit_rate) else min_hit_rate)
        + 500.0 * (0 if np.isnan(mean_hit_rate) else mean_hit_rate)
        - 100000.0 * max(0.0, one_day_dependency - 0.50)
    )

    summary = {
        "candidate": spec["name"],
        "family": spec.get("family", spec["kind"]),
        "kind": spec["kind"],
        "note": spec.get("note", ""),
        **cfg,
        "days": int(day_df["day"].nunique()),
        "total_pnl": total_pnl,
        "mean_day_pnl": mean_day_pnl,
        "min_day_pnl": min_day_pnl,
        "max_day_pnl": max_day_pnl,
        "total_trades": total_trades,
        "mean_hit_rate": mean_hit_rate,
        "min_hit_rate": min_hit_rate,
        "pnl_per_trade": pnl_per_trade,
        "one_day_dependency": one_day_dependency,
        "robust_pass": robust_pass,
        "robust_score": robust_score,
    }

    return summary, day_df


def run_config(spec: dict, window, entry_z, exit_z, max_hold, gross, exit_mode):
    cfg = {
        "window": window,
        "entry_z": entry_z,
        "exit_z": exit_z,
        "exit_mode": exit_mode,
        "max_hold": max_hold,
        "gross": gross,
    }

    all_trades = []
    for day, mid_df in sorted(uv_by_day.items()):
        day_trades = strict_backtest_one_day(
            mid_df=mid_df,
            spec=spec,
            window=window,
            entry_z=entry_z,
            exit_z=exit_z,
            max_hold=max_hold,
            gross=gross,
            exit_mode=exit_mode,
            day=day,
        )
        all_trades.extend(day_trades)

    trades_df = pd.DataFrame(all_trades)
    summary, day_df = summarize_trades(trades_df, spec, cfg)

    return summary, day_df, trades_df


def run_grid(specs, windows, entry_zs, exit_zs_by_mode, max_holds, grosses, label="grid"):
    rows = []
    day_rows = []
    trade_rows = []

    configs = []
    for spec in specs:
        for window, entry_z, max_hold, gross, exit_mode in itertools.product(
            windows, entry_zs, max_holds, grosses, exit_zs_by_mode.keys()
        ):
            for exit_z in exit_zs_by_mode[exit_mode]:
                configs.append((spec, window, entry_z, exit_z, max_hold, gross, exit_mode))

    print(f"{label}: total configs = {len(configs)}")
    t0 = time.time()

    for k, (spec, window, entry_z, exit_z, max_hold, gross, exit_mode) in enumerate(configs, 1):
        if k == 1 or k % 250 == 0 or k == len(configs):
            elapsed = time.time() - t0
            print(
                f"[{elapsed:7.2f}s] {label} {k}/{len(configs)}: "
                f"{spec['name']}, w={window}, entry={entry_z}, exit={exit_z}, "
                f"mode={exit_mode}, hold={max_hold}, gross={gross}"
            )

        summary, day_df, trades_df = run_config(
            spec=spec,
            window=window,
            entry_z=entry_z,
            exit_z=exit_z,
            max_hold=max_hold,
            gross=gross,
            exit_mode=exit_mode,
        )

        rows.append(summary)
        day_rows.append(day_df)

        if not trades_df.empty:
            trade_rows.append(trades_df)

    summary_df = pd.DataFrame(rows).sort_values(
        ["robust_pass", "robust_score", "total_pnl"],
        ascending=[False, False, False],
    )

    day_breakdown_df = pd.concat(day_rows, ignore_index=True) if day_rows else pd.DataFrame()
    trades_all_df = pd.concat(trade_rows, ignore_index=True) if trade_rows else pd.DataFrame()

    return summary_df, day_breakdown_df, trades_all_df

In [16]:
LEAD_LAG_SPECS = [
    {
        "name": "lead_AMBER_to_RED_inverse_lag1000",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["AMBER"],
        "follower": COLOUR_TO_PRODUCT["RED"],
        "relation": "inverse",
        "lag": 1000,
        "note": "RED inversely follows AMBER return over lag 1000",
    },
    {
        "name": "lead_RED_to_YELLOW_inverse_lag250",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["RED"],
        "follower": COLOUR_TO_PRODUCT["YELLOW"],
        "relation": "inverse",
        "lag": 250,
        "note": "YELLOW inversely follows RED return over lag 250",
    },
    {
        "name": "lead_RED_to_YELLOW_inverse_lag1000",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["RED"],
        "follower": COLOUR_TO_PRODUCT["YELLOW"],
        "relation": "inverse",
        "lag": 1000,
        "note": "YELLOW inversely follows RED return over lag 1000",
    },
    {
        "name": "lead_YELLOW_to_RED_follow_lag1000",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["YELLOW"],
        "follower": COLOUR_TO_PRODUCT["RED"],
        "relation": "follow",
        "lag": 1000,
        "note": "RED follows YELLOW return over lag 1000",
    },
    {
        "name": "lead_AMBER_to_ORANGE_follow_lag50",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["AMBER"],
        "follower": COLOUR_TO_PRODUCT["ORANGE"],
        "relation": "follow",
        "lag": 50,
        "note": "ORANGE follows AMBER return over lag 50",
    },
    {
        "name": "lead_AMBER_to_ORANGE_follow_lag100",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["AMBER"],
        "follower": COLOUR_TO_PRODUCT["ORANGE"],
        "relation": "follow",
        "lag": 100,
        "note": "ORANGE follows AMBER return over lag 100",
    },
    {
        "name": "lead_ORANGE_to_YELLOW_follow_lag100",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["ORANGE"],
        "follower": COLOUR_TO_PRODUCT["YELLOW"],
        "relation": "follow",
        "lag": 100,
        "note": "YELLOW follows ORANGE return over lag 100",
    },
    {
        "name": "lead_AMBER_to_YELLOW_inverse_lag100",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["AMBER"],
        "follower": COLOUR_TO_PRODUCT["YELLOW"],
        "relation": "inverse",
        "lag": 100,
        "note": "YELLOW inversely follows AMBER return over lag 100",
    },
    {
        "name": "lead_MAGENTA_to_YELLOW_inverse_lag100",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["MAGENTA"],
        "follower": COLOUR_TO_PRODUCT["YELLOW"],
        "relation": "inverse",
        "lag": 100,
        "note": "YELLOW inversely follows MAGENTA return over lag 100",
    },
    {
        "name": "lead_MAGENTA_to_YELLOW_follow_lag100",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["MAGENTA"],
        "follower": COLOUR_TO_PRODUCT["YELLOW"],
        "relation": "follow",
        "lag": 100,
        "note": "YELLOW follows MAGENTA return over lag 100",
    },
    {
        "name": "lead_AMBER_to_MAGENTA_inverse_lag50",
        "kind": "lead_lag",
        "family": "lead_lag",
        "leader": COLOUR_TO_PRODUCT["AMBER"],
        "follower": COLOUR_TO_PRODUCT["MAGENTA"],
        "relation": "inverse",
        "lag": 50,
        "note": "MAGENTA inversely follows AMBER return over lag 50",
    },
]


MIXING_SPECS = [
    {
        "name": "mix_RED_blend_YELLOW_MAGENTA_75_25_mr",
        "kind": "mix_mr",
        "family": "colour_mixing",
        "target": COLOUR_TO_PRODUCT["RED"],
        "a": COLOUR_TO_PRODUCT["YELLOW"],
        "b": COLOUR_TO_PRODUCT["MAGENTA"],
        "wa": 0.75,
        "note": "RED residual vs 75% YELLOW + 25% MAGENTA",
    },
    {
        "name": "mix_RED_blend_YELLOW_MAGENTA_50_50_mr",
        "kind": "mix_mr",
        "family": "colour_mixing",
        "target": COLOUR_TO_PRODUCT["RED"],
        "a": COLOUR_TO_PRODUCT["YELLOW"],
        "b": COLOUR_TO_PRODUCT["MAGENTA"],
        "wa": 0.50,
        "note": "RED residual vs 50% YELLOW + 50% MAGENTA",
    },
    {
        "name": "mix_ORANGE_blend_YELLOW_RED_50_50_mr",
        "kind": "mix_mr",
        "family": "colour_mixing",
        "target": COLOUR_TO_PRODUCT["ORANGE"],
        "a": COLOUR_TO_PRODUCT["YELLOW"],
        "b": COLOUR_TO_PRODUCT["RED"],
        "wa": 0.50,
        "note": "ORANGE residual vs 50% YELLOW + 50% RED",
    },
    {
        "name": "mix_AMBER_blend_YELLOW_ORANGE_50_50_mr",
        "kind": "mix_mr",
        "family": "colour_mixing",
        "target": COLOUR_TO_PRODUCT["AMBER"],
        "a": COLOUR_TO_PRODUCT["YELLOW"],
        "b": COLOUR_TO_PRODUCT["ORANGE"],
        "wa": 0.50,
        "note": "AMBER residual vs 50% YELLOW + 50% ORANGE",
    },
]


PAIRWISE_SPECS = [
    {
        "name": "YELLOW_vs_ORANGE_ratio_mr",
        "kind": "pair_ratio",
        "family": "pairwise",
        "a": COLOUR_TO_PRODUCT["YELLOW"],
        "b": COLOUR_TO_PRODUCT["ORANGE"],
        "note": "YELLOW / ORANGE ratio mean reversion",
    },
    {
        "name": "YELLOW_vs_ORANGE_spread_mr",
        "kind": "pair_spread",
        "family": "pairwise",
        "a": COLOUR_TO_PRODUCT["YELLOW"],
        "b": COLOUR_TO_PRODUCT["ORANGE"],
        "note": "YELLOW - ORANGE spread mean reversion",
    },
]

ALL_VALIDATION_SPECS = LEAD_LAG_SPECS + MIXING_SPECS + PAIRWISE_SPECS

In [17]:
best_spec = LEAD_LAG_SPECS[0]  # lead_AMBER_to_RED_inverse_lag1000

EXIT_VALIDATION_MODES = {
    "fixed_hold": [np.nan],
    "zero_cross": [0.0],
    "abs_band": [0.25, 0.50, 0.75, 1.00],
    "signal_flip": [np.nan],
}

exit_validation_summary, exit_validation_days, exit_validation_trades = run_grid(
    specs=[best_spec],
    windows=[500],
    entry_zs=[1.50],
    exit_zs_by_mode=EXIT_VALIDATION_MODES,
    max_holds=[2500],
    grosses=[80],
    label="exit_validation_best",
)

exit_validation_summary.to_csv(OUT_DIR / "exit_validation_best_summary.csv", index=False)
exit_validation_days.to_csv(OUT_DIR / "exit_validation_best_days.csv", index=False)
exit_validation_trades.to_csv(OUT_DIR / "exit_validation_best_trades.csv", index=False)

display(exit_validation_summary)
display(exit_validation_days.sort_values(["exit_mode", "exit_z", "day"]))
display(exit_validation_trades.sort_values(["exit_mode", "exit_z", "day", "entry_idx"]).head(100))

exit_validation_best: total configs = 7
[   0.00s] exit_validation_best 1/7: lead_AMBER_to_RED_inverse_lag1000, w=500, entry=1.5, exit=nan, mode=fixed_hold, hold=2500, gross=80
[   0.08s] exit_validation_best 7/7: lead_AMBER_to_RED_inverse_lag1000, w=500, entry=1.5, exit=nan, mode=signal_flip, hold=2500, gross=80


,candidate,family,kind,note,window,entry_z,exit_z,exit_mode,max_hold,gross,days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
6,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,500,1.5,NaN,signal_flip,2500,80,3,272960.0,90986.666667,39080.0,163000.0,46,0.589169,0.529412,5933.913043,0.597157,True,344345.934063
0,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,500,1.5,NaN,fixed_hold,2500,80,3,255760.0,85253.333333,-17240.0,142160.0,12,0.666667,0.250000,21313.333333,0.555834,False,217279.973934
1,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,500,1.5,0.00,zero_cross,2500,80,3,138360.0,46120.000000,-16680.0,100240.0,69,0.508485,0.480000,2005.217391,0.724487,False,85205.557833
2,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,500,1.5,0.25,abs_band,2500,80,3,112960.0,37653.333333,-23160.0,83560.0,72,0.498328,0.478261,1568.888889,0.739731,False,45307.380409
5,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,500,1.5,1.00,abs_band,2500,80,3,91520.0,30506.666667,-17720.0,87720.0,128,0.557787,0.477273,715.000000,0.958479,False,12897.355277
4,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,500,1.5,0.75,abs_band,2500,80,3,75920.0,25306.666667,-21520.0,64640.0,104,0.528644,0.457143,730.000000,0.851423,False,287.781070
3,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,500,1.5,0.50,abs_band,2500,80,3,66880.0,22293.333333,-19800.0,57840.0,85,0.494679,0.448276,786.823529,0.864833,False,-6714.534985


,day,trade_count,day_pnl,hit_rate,avg_trade_pnl,median_trade_pnl,window,entry_z,exit_z,exit_mode,max_hold,gross,candidate,family,kind,note
6,2,23,52560.0,0.478261,2285.217391,-1280.0,500,1.5,0.25,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
7,3,23,-23160.0,0.478261,-1006.956522,-800.0,500,1.5,0.25,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
8,4,26,83560.0,0.538462,3213.846154,980.0,500,1.5,0.25,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
9,2,29,28840.0,0.448276,994.482759,-1440.0,500,1.5,0.50,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
10,3,27,-19800.0,0.518519,-733.333333,1760.0,500,1.5,0.50,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
11,4,29,57840.0,0.517241,1994.482759,320.0,500,1.5,0.50,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
12,2,35,32800.0,0.457143,937.142857,-1000.0,500,1.5,0.75,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
13,3,36,-21520.0,0.583333,-597.777778,1640.0,500,1.5,0.75,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
14,4,33,64640.0,0.545455,1958.787879,200.0,500,1.5,0.75,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000
15,2,44,21520.0,0.477273,489.090909,-240.0,500,1.5,1.00,abs_band,2500,80,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000


,candidate,family,kind,note,day,window,entry_z,exit_z,exit_mode,max_hold,gross,q_trade,net_position,gross_position,entry_idx,exit_idx,entry_ts,exit_ts,side,entry_z_seen,exit_z_seen,hold,exec_pnl,position
81,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.25,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,1840,1874,184000,187400,-1,2.020327,0.218677,34,6560.0,"[-0.0, -0.0, -0.0, -80.0, -0.0]"
82,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.25,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,1892,1906,189200,190600,-1,1.579071,0.186395,14,-1280.0,"[-0.0, -0.0, -0.0, -80.0, -0.0]"
83,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.25,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,1936,2045,193600,204500,1,-1.559829,-0.242369,109,8440.0,"[0.0, 0.0, 0.0, 80.0, 0.0]"
84,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.25,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,2117,2132,211700,213200,-1,1.506158,0.243850,15,-5840.0,"[-0.0, -0.0, -0.0, -80.0, -0.0]"
85,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.25,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,2178,2225,217800,222500,1,-1.728200,-0.210113,47,-6240.0,"[0.0, 0.0, 0.0, 80.0, 0.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.50,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,7377,7487,737700,748700,1,-1.857883,-0.482609,110,16360.0,"[0.0, 0.0, 0.0, 80.0, 0.0]"
177,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.50,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,7629,7651,762900,765100,1,-1.581905,-0.413650,22,2280.0,"[0.0, 0.0, 0.0, 80.0, 0.0]"
178,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.50,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,7715,7954,771500,795400,-1,1.738430,0.487093,239,3000.0,"[-0.0, -0.0, -0.0, -80.0, -0.0]"
179,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,2,500,1.5,0.50,abs_band,2500,80,"[0.0, 0.0, 0.0, 80.0, 0.0]",80.0,80.0,8310,8692,831000,869200,-1,1.584429,0.425318,382,25120.0,"[-0.0, -0.0, -0.0, -80.0, -0.0]"


In [19]:
cols = [
    "candidate", "exit_mode", "exit_z", "max_hold",
    "total_pnl", "min_day_pnl", "max_day_pnl",
    "total_trades", "mean_hit_rate", "one_day_dependency",
    "robust_pass", "robust_score",
]

display(exit_validation_summary[cols].sort_values("robust_score", ascending=False))

,candidate,exit_mode,exit_z,max_hold,total_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,one_day_dependency,robust_pass,robust_score
6,lead_AMBER_to_RED_inverse_lag1000,signal_flip,NaN,2500,272960.0,39080.0,163000.0,46,0.589169,0.597157,True,344345.934063
0,lead_AMBER_to_RED_inverse_lag1000,fixed_hold,NaN,2500,255760.0,-17240.0,142160.0,12,0.666667,0.555834,False,217279.973934
1,lead_AMBER_to_RED_inverse_lag1000,zero_cross,0.00,2500,138360.0,-16680.0,100240.0,69,0.508485,0.724487,False,85205.557833
2,lead_AMBER_to_RED_inverse_lag1000,abs_band,0.25,2500,112960.0,-23160.0,83560.0,72,0.498328,0.739731,False,45307.380409
5,lead_AMBER_to_RED_inverse_lag1000,abs_band,1.00,2500,91520.0,-17720.0,87720.0,128,0.557787,0.958479,False,12897.355277
4,lead_AMBER_to_RED_inverse_lag1000,abs_band,0.75,2500,75920.0,-21520.0,64640.0,104,0.528644,0.851423,False,287.781070
3,lead_AMBER_to_RED_inverse_lag1000,abs_band,0.50,2500,66880.0,-19800.0,57840.0,85,0.494679,0.864833,False,-6714.534985


In [20]:
DISCOVERED_TOP_CONFIGS = [
    # spec_name, window, entry_z, max_hold, gross
    ("lead_AMBER_to_RED_inverse_lag1000", 500, 1.50, 2500, 80),
    ("lead_RED_to_YELLOW_inverse_lag250", 500, 2.00, 1500, 80),
    ("lead_RED_to_YELLOW_inverse_lag1000", 500, 1.75, 1500, 80),
    ("lead_YELLOW_to_RED_follow_lag1000", 500, 1.25, 1000, 80),
    ("lead_AMBER_to_ORANGE_follow_lag50", 500, 1.50, 1500, 80),
    ("lead_AMBER_to_ORANGE_follow_lag100", 500, 1.25, 1500, 80),
    ("lead_ORANGE_to_YELLOW_follow_lag100", 1000, 1.75, 500, 80),
    ("lead_AMBER_to_YELLOW_inverse_lag100", 1000, 1.75, 1500, 80),
    ("lead_MAGENTA_to_YELLOW_inverse_lag100", 500, 2.25, 4000, 80),
    ("lead_MAGENTA_to_YELLOW_follow_lag100", 500, 1.50, 500, 80),
    ("lead_AMBER_to_MAGENTA_inverse_lag50", 1000, 2.00, 1000, 80),
]

spec_by_name = {s["name"]: s for s in ALL_VALIDATION_SPECS}

rows = []
day_rows = []
trade_rows = []

for spec_name, window, entry_z, max_hold, gross in DISCOVERED_TOP_CONFIGS:
    spec = spec_by_name[spec_name]
    for exit_mode, exit_zs in EXIT_VALIDATION_MODES.items():
        for exit_z in exit_zs:
            summary, days, trades = run_config(
                spec=spec,
                window=window,
                entry_z=entry_z,
                exit_z=exit_z,
                max_hold=max_hold,
                gross=gross,
                exit_mode=exit_mode,
            )
            rows.append(summary)
            day_rows.append(days)
            if not trades.empty:
                trade_rows.append(trades)

top_exit_compare_summary = pd.DataFrame(rows).sort_values(
    ["robust_pass", "robust_score", "total_pnl"],
    ascending=[False, False, False],
)

top_exit_compare_days = pd.concat(day_rows, ignore_index=True)
top_exit_compare_trades = pd.concat(trade_rows, ignore_index=True) if trade_rows else pd.DataFrame()

top_exit_compare_summary.to_csv(OUT_DIR / "top_exit_compare_summary.csv", index=False)
top_exit_compare_days.to_csv(OUT_DIR / "top_exit_compare_days.csv", index=False)
top_exit_compare_trades.to_csv(OUT_DIR / "top_exit_compare_trades.csv", index=False)

display(top_exit_compare_summary[cols].head(80))

,candidate,exit_mode,exit_z,max_hold,total_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,one_day_dependency,robust_pass,robust_score
7,lead_RED_to_YELLOW_inverse_lag250,fixed_hold,NaN,1500,426960.0,105120.0,184160.0,18,0.888889,0.431328,True,6.418111e+05
63,lead_MAGENTA_to_YELLOW_follow_lag100,fixed_hold,NaN,500,370200.0,113280.0,138480.0,50,0.722222,0.374068,True,6.004544e+05
21,lead_YELLOW_to_RED_follow_lag1000,fixed_hold,NaN,1000,448840.0,68120.0,191040.0,27,0.851852,0.425631,True,5.893948e+05
76,lead_AMBER_to_MAGENTA_inverse_lag50,signal_flip,NaN,1000,382000.0,99920.0,150640.0,63,0.603319,0.394346,True,5.850962e+05
20,lead_RED_to_YELLOW_inverse_lag1000,signal_flip,NaN,1500,419800.0,78400.0,215320.0,37,0.604978,0.512911,True,5.777543e+05
42,lead_ORANGE_to_YELLOW_follow_lag100,fixed_hold,NaN,500,353760.0,91560.0,134560.0,42,0.669109,0.380371,True,5.400717e+05
35,lead_AMBER_to_ORANGE_follow_lag100,fixed_hold,NaN,1500,349960.0,90080.0,166640.0,21,0.666667,0.476169,True,5.333105e+05
56,lead_MAGENTA_to_YELLOW_inverse_lag100,fixed_hold,NaN,4000,356120.0,84160.0,169400.0,9,0.777778,0.475682,True,5.281622e+05
70,lead_AMBER_to_MAGENTA_inverse_lag50,fixed_hold,NaN,1000,361080.0,76760.0,155240.0,24,0.705688,0.429932,True,5.180778e+05
49,lead_AMBER_to_YELLOW_inverse_lag100,fixed_hold,NaN,1500,316080.0,95640.0,113720.0,16,0.744444,0.359782,True,5.107322e+05


In [21]:
pivot_exit = top_exit_compare_summary.pivot_table(
    index="candidate",
    columns=["exit_mode", "exit_z"],
    values="total_pnl",
    aggfunc="max",
)

display(pivot_exit)

exit_mode                              abs_band                               zero_cross
exit_z                                     0.25      0.50      0.75      1.00       0.00
candidate                                                                               
lead_AMBER_to_MAGENTA_inverse_lag50    165600.0   90360.0   84240.0   59360.0   170320.0
lead_AMBER_to_ORANGE_follow_lag100     -17880.0  -14520.0    7360.0   -4600.0   -16560.0
lead_AMBER_to_ORANGE_follow_lag50      -15240.0  -13760.0  -10680.0  -40200.0    -1000.0
lead_AMBER_to_RED_inverse_lag1000      112960.0   66880.0   75920.0   91520.0   138360.0
lead_AMBER_to_YELLOW_inverse_lag100     78600.0   56480.0   43960.0   29560.0   121000.0
lead_MAGENTA_to_YELLOW_follow_lag100   167800.0  101200.0   98000.0   69240.0   125600.0
lead_MAGENTA_to_YELLOW_inverse_lag100  -90880.0  -95920.0  -63120.0  -59200.0   -56120.0
lead_ORANGE_to_YELLOW_follow_lag100     -4960.0  -36520.0  -51080.0  -14440.0   -47800.0
lead_RED_to_YELLOW_inverse_lag1000     225320.0  201320.0  234320.0  152680.0   264160.0
lead_RED_to_YELLOW_inverse_lag250        3480.0   26800.0    8680.0   -4600.0     2520.0
lead_YELLOW_to_RED_follow_lag1000      278920.0  281000.0  270440.0  303360.0   284960.0

In [22]:
FOCUSED_EXIT_MODES = {
    "fixed_hold": [np.nan],
    "abs_band": [0.25, 0.50, 0.75],
    "zero_cross": [0.0],
}

focused_summary, focused_days, focused_trades = run_grid(
    specs=LEAD_LAG_SPECS,
    windows=[250, 500, 750, 1000, 1500],
    entry_zs=[1.25, 1.50, 1.75, 2.00, 2.25],
    exit_zs_by_mode=FOCUSED_EXIT_MODES,
    max_holds=[500, 1000, 1500, 2500, 4000],
    grosses=[60, 72, 80],
    label="focused_lead_lag",
)

focused_summary.to_csv(OUT_DIR / "focused_lead_lag_summary.csv", index=False)
focused_days.to_csv(OUT_DIR / "focused_lead_lag_days.csv", index=False)
focused_trades.to_csv(OUT_DIR / "focused_lead_lag_trades.csv", index=False)

display(focused_summary.head(80))

focused_lead_lag: total configs = 20625
[   0.00s] focused_lead_lag 1/20625: lead_AMBER_to_RED_inverse_lag1000, w=250, entry=1.25, exit=nan, mode=fixed_hold, hold=500, gross=60
[   2.11s] focused_lead_lag 250/20625: lead_AMBER_to_RED_inverse_lag1000, w=250, entry=2.0, exit=0.0, mode=zero_cross, hold=1000, gross=72
[   4.05s] focused_lead_lag 500/20625: lead_AMBER_to_RED_inverse_lag1000, w=500, entry=1.5, exit=0.0, mode=zero_cross, hold=2500, gross=60
[   6.06s] focused_lead_lag 750/20625: lead_AMBER_to_RED_inverse_lag1000, w=500, entry=2.25, exit=0.0, mode=zero_cross, hold=4000, gross=80
[   7.97s] focused_lead_lag 1000/20625: lead_AMBER_to_RED_inverse_lag1000, w=750, entry=2.0, exit=0.0, mode=zero_cross, hold=1000, gross=72
[   9.83s] focused_lead_lag 1250/20625: lead_AMBER_to_RED_inverse_lag1000, w=1000, entry=1.5, exit=0.0, mode=zero_cross, hold=2500, gross=60
[  11.74s] focused_lead_lag 1500/20625: lead_AMBER_to_RED_inverse_lag1000, w=1000, entry=2.25, exit=0.0, mode=zero_cross, ho

,candidate,family,kind,note,window,entry_z,exit_z,exit_mode,max_hold,gross,days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
2590,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,1500,80,3,443440.0,147813.333333,122160.0,164760.0,17,0.822222,0.800000,26084.705882,0.371550,True,692171.111111
2515,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.00,NaN,fixed_hold,1500,80,3,426960.0,142320.000000,105120.0,184160.0,18,0.888889,0.833333,23720.000000,0.431328,True,641811.111111
2585,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,1500,72,3,399096.0,133032.000000,109944.0,148284.0,17,0.822222,0.800000,23476.235294,0.371550,True,623395.111111
4195,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,500,1.25,NaN,fixed_hold,4000,80,3,386360.0,128786.666667,108440.0,139520.0,8,0.888889,0.666667,48295.000000,0.361114,True,607017.777778
2620,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,4000,80,3,411920.0,137306.666667,92840.0,167120.0,9,0.888889,0.666667,45768.888889,0.405710,True,601377.777778
17335,lead_MAGENTA_to_YELLOW_follow_lag100,lead_lag,lead_lag,YELLOW follows MAGENTA return over lag 100,500,1.50,NaN,fixed_hold,500,80,3,370200.0,123400.000000,113280.0,138480.0,50,0.722222,0.666667,7404.000000,0.374068,True,600454.444444
18910,lead_AMBER_to_MAGENTA_inverse_lag50,lead_lag,lead_lag,MAGENTA inversely follows AMBER return over la...,250,1.75,NaN,fixed_hold,500,80,3,386680.0,128893.333333,105000.0,160760.0,54,0.597007,0.500000,7160.740741,0.415744,True,599478.503612
9970,lead_AMBER_to_ORANGE_follow_lag100,lead_lag,lead_lag,ORANGE follows AMBER return over lag 100,500,1.75,NaN,fixed_hold,4000,80,3,438280.0,146093.333333,74200.0,214080.0,9,0.888889,0.666667,48697.777778,0.488455,True,590457.777778
6025,lead_YELLOW_to_RED_follow_lag1000,lead_lag,lead_lag,RED follows YELLOW return over lag 1000,500,1.25,NaN,fixed_hold,1000,80,3,448840.0,149613.333333,68120.0,191040.0,27,0.851852,0.777778,16623.703704,0.425631,True,589394.814815
1180,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,1000,1.25,NaN,fixed_hold,2500,80,3,370280.0,123426.666667,107360.0,140840.0,11,0.805556,0.666667,33661.818182,0.380361,True,588736.111111


In [23]:
display(
    focused_summary.sort_values(
        ["robust_pass", "robust_score", "min_day_pnl", "total_pnl"],
        ascending=[False, False, False, False],
    ).head(50)
)

,candidate,family,kind,note,window,entry_z,exit_z,exit_mode,max_hold,gross,days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
2590,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,1500,80,3,443440.0,147813.333333,122160.0,164760.0,17,0.822222,0.800000,26084.705882,0.371550,True,692171.111111
2515,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.00,NaN,fixed_hold,1500,80,3,426960.0,142320.000000,105120.0,184160.0,18,0.888889,0.833333,23720.000000,0.431328,True,641811.111111
2585,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,1500,72,3,399096.0,133032.000000,109944.0,148284.0,17,0.822222,0.800000,23476.235294,0.371550,True,623395.111111
4195,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,500,1.25,NaN,fixed_hold,4000,80,3,386360.0,128786.666667,108440.0,139520.0,8,0.888889,0.666667,48295.000000,0.361114,True,607017.777778
2620,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,4000,80,3,411920.0,137306.666667,92840.0,167120.0,9,0.888889,0.666667,45768.888889,0.405710,True,601377.777778
17335,lead_MAGENTA_to_YELLOW_follow_lag100,lead_lag,lead_lag,YELLOW follows MAGENTA return over lag 100,500,1.50,NaN,fixed_hold,500,80,3,370200.0,123400.000000,113280.0,138480.0,50,0.722222,0.666667,7404.000000,0.374068,True,600454.444444
18910,lead_AMBER_to_MAGENTA_inverse_lag50,lead_lag,lead_lag,MAGENTA inversely follows AMBER return over la...,250,1.75,NaN,fixed_hold,500,80,3,386680.0,128893.333333,105000.0,160760.0,54,0.597007,0.500000,7160.740741,0.415744,True,599478.503612
9970,lead_AMBER_to_ORANGE_follow_lag100,lead_lag,lead_lag,ORANGE follows AMBER return over lag 100,500,1.75,NaN,fixed_hold,4000,80,3,438280.0,146093.333333,74200.0,214080.0,9,0.888889,0.666667,48697.777778,0.488455,True,590457.777778
6025,lead_YELLOW_to_RED_follow_lag1000,lead_lag,lead_lag,RED follows YELLOW return over lag 1000,500,1.25,NaN,fixed_hold,1000,80,3,448840.0,149613.333333,68120.0,191040.0,27,0.851852,0.777778,16623.703704,0.425631,True,589394.814815
1180,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,1000,1.25,NaN,fixed_hold,2500,80,3,370280.0,123426.666667,107360.0,140840.0,11,0.805556,0.666667,33661.818182,0.380361,True,588736.111111


In [24]:
display(
    focused_summary.sort_values(
        ["total_pnl", "min_day_pnl"],
        ascending=[False, False],
    ).head(50)
)

,candidate,family,kind,note,window,entry_z,exit_z,exit_mode,max_hold,gross,days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
6025,lead_YELLOW_to_RED_follow_lag1000,lead_lag,lead_lag,RED follows YELLOW return over lag 1000,500,1.25,NaN,fixed_hold,1000,80,3,448840.0,149613.333333,68120.0,191040.0,27,0.851852,0.777778,16623.703704,0.425631,True,589394.814815
2590,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,1500,80,3,443440.0,147813.333333,122160.0,164760.0,17,0.822222,0.800000,26084.705882,0.371550,True,692171.111111
9970,lead_AMBER_to_ORANGE_follow_lag100,lead_lag,lead_lag,ORANGE follows AMBER return over lag 100,500,1.75,NaN,fixed_hold,4000,80,3,438280.0,146093.333333,74200.0,214080.0,9,0.888889,0.666667,48697.777778,0.488455,True,590457.777778
2515,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.00,NaN,fixed_hold,1500,80,3,426960.0,142320.000000,105120.0,184160.0,18,0.888889,0.833333,23720.000000,0.431328,True,641811.111111
4886,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,1000,1.25,0.25,abs_band,500,80,3,426360.0,142120.000000,75720.0,220400.0,66,0.702648,0.631579,6460.000000,0.516934,True,579615.813866
2620,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,4000,80,3,411920.0,137306.666667,92840.0,167120.0,9,0.888889,0.666667,45768.888889,0.405710,True,601377.777778
6100,lead_YELLOW_to_RED_follow_lag1000,lead_lag,lead_lag,RED follows YELLOW return over lag 1000,500,1.50,NaN,fixed_hold,1000,80,3,411360.0,137120.000000,57240.0,192680.0,26,0.773148,0.666667,15821.538462,0.468398,True,529559.907407
4885,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,1000,1.25,NaN,fixed_hold,500,80,3,408480.0,136160.000000,89600.0,224440.0,42,0.633333,0.500000,9725.714286,0.549452,True,585551.504113
6020,lead_YELLOW_to_RED_follow_lag1000,lead_lag,lead_lag,RED follows YELLOW return over lag 1000,500,1.25,NaN,fixed_hold,1000,72,3,403956.0,134652.000000,61308.0,171936.0,27,0.851852,0.777778,14961.333333,0.425631,True,530886.814815
4889,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,1000,1.25,0.00,zero_cross,500,80,3,401360.0,133786.666667,71320.0,229400.0,58,0.612979,0.500000,6920.000000,0.571557,True,539650.818543


In [25]:
display(
    focused_summary.sort_values(
        ["min_day_pnl", "total_pnl"],
        ascending=[False, False],
    ).head(50)
)

,candidate,family,kind,note,window,entry_z,exit_z,exit_mode,max_hold,gross,days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
2590,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,1500,80,3,443440.0,147813.333333,122160.0,164760.0,17,0.822222,0.800000,26084.705882,0.371550,True,692171.111111
17335,lead_MAGENTA_to_YELLOW_follow_lag100,lead_lag,lead_lag,YELLOW follows MAGENTA return over lag 100,500,1.50,NaN,fixed_hold,500,80,3,370200.0,123400.000000,113280.0,138480.0,50,0.722222,0.666667,7404.000000,0.374068,True,600454.444444
2585,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,1500,72,3,399096.0,133032.000000,109944.0,148284.0,17,0.822222,0.800000,23476.235294,0.371550,True,623395.111111
4195,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,500,1.25,NaN,fixed_hold,4000,80,3,386360.0,128786.666667,108440.0,139520.0,8,0.888889,0.666667,48295.000000,0.361114,True,607017.777778
1180,lead_AMBER_to_RED_inverse_lag1000,lead_lag,lead_lag,RED inversely follows AMBER return over lag 1000,1000,1.25,NaN,fixed_hold,2500,80,3,370280.0,123426.666667,107360.0,140840.0,11,0.805556,0.666667,33661.818182,0.380361,True,588736.111111
2515,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.00,NaN,fixed_hold,1500,80,3,426960.0,142320.000000,105120.0,184160.0,18,0.888889,0.833333,23720.000000,0.431328,True,641811.111111
18910,lead_AMBER_to_MAGENTA_inverse_lag50,lead_lag,lead_lag,MAGENTA inversely follows AMBER return over la...,250,1.75,NaN,fixed_hold,500,80,3,386680.0,128893.333333,105000.0,160760.0,54,0.597007,0.500000,7160.740741,0.415744,True,599478.503612
4754,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,750,2.00,0.00,zero_cross,1000,80,3,361360.0,120453.333333,104480.0,138680.0,34,0.717236,0.555556,10628.235294,0.383772,True,573456.396011
4769,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,750,2.00,0.00,zero_cross,1500,80,3,353920.0,117973.333333,104480.0,138680.0,34,0.717236,0.555556,10409.411765,0.391840,True,566016.396011
4784,lead_RED_to_YELLOW_inverse_lag1000,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 1000,750,2.00,0.00,zero_cross,2500,80,3,353920.0,117973.333333,104480.0,138680.0,34,0.717236,0.555556,10409.411765,0.391840,True,566016.396011


In [26]:
exit_mode_comparison = (
    focused_summary
    .groupby("exit_mode")
    .agg(
        best_total_pnl=("total_pnl", "max"),
        best_min_day_pnl=("min_day_pnl", "max"),
        best_robust_score=("robust_score", "max"),
        robust_count=("robust_pass", "sum"),
        configs=("candidate", "size"),
    )
    .sort_values("best_robust_score", ascending=False)
)

display(exit_mode_comparison)

candidate_exit_mode_comparison = (
    focused_summary
    .groupby(["candidate", "exit_mode"])
    .agg(
        best_total_pnl=("total_pnl", "max"),
        best_min_day_pnl=("min_day_pnl", "max"),
        best_robust_score=("robust_score", "max"),
        robust_count=("robust_pass", "sum"),
        configs=("candidate", "size"),
    )
    .reset_index()
    .sort_values(["candidate", "best_robust_score"], ascending=[True, False])
)

display(candidate_exit_mode_comparison)

,best_total_pnl,best_min_day_pnl,best_robust_score,robust_count,configs
exit_mode,,,,,
fixed_hold,448840.0,122160.0,692171.111111,666,4125
zero_cross,401360.0,104480.0,586006.056166,1305,4125
abs_band,426360.0,98200.0,579615.813866,3423,12375


,candidate,exit_mode,best_total_pnl,best_min_day_pnl,best_robust_score,robust_count,configs
1,lead_AMBER_to_MAGENTA_inverse_lag50,fixed_hold,386680.0,105000.0,5.994785e+05,87,375
2,lead_AMBER_to_MAGENTA_inverse_lag50,zero_cross,257280.0,58520.0,3.640026e+05,285,375
0,lead_AMBER_to_MAGENTA_inverse_lag50,abs_band,208760.0,57080.0,3.227470e+05,630,1125
4,lead_AMBER_to_ORANGE_follow_lag100,fixed_hold,438280.0,90080.0,5.904578e+05,27,375
5,lead_AMBER_to_ORANGE_follow_lag100,zero_cross,92000.0,-6870.0,5.683801e+04,0,375
3,lead_AMBER_to_ORANGE_follow_lag100,abs_band,64240.0,-10680.0,-4.578836e+04,0,1125
7,lead_AMBER_to_ORANGE_follow_lag50,fixed_hold,384920.0,85520.0,5.094422e+05,36,375
6,lead_AMBER_to_ORANGE_follow_lag50,abs_band,51960.0,3040.0,4.838678e+04,15,1125
8,lead_AMBER_to_ORANGE_follow_lag50,zero_cross,57960.0,3600.0,1.471072e+04,0,375
10,lead_AMBER_to_RED_inverse_lag1000,fixed_hold,370280.0,107360.0,5.887361e+05,57,375


In [27]:
fixed_hold_wins = (
    focused_summary
    .sort_values("robust_score", ascending=False)
    .head(100)
    ["exit_mode"]
    .value_counts()
)

display(fixed_hold_wins)

exit_mode
fixed_hold    43
zero_cross    29
abs_band      28
Name: count, dtype: int64

In [28]:
best_row = focused_summary.sort_values(
    ["robust_pass", "robust_score", "total_pnl"],
    ascending=[False, False, False],
).iloc[0]

display(best_row.to_frame().T)

mask = (
    (focused_days["candidate"] == best_row["candidate"])
    & (focused_days["window"] == best_row["window"])
    & (focused_days["entry_z"] == best_row["entry_z"])
    & (focused_days["exit_mode"] == best_row["exit_mode"])
    & (
        (focused_days["exit_z"].isna() & pd.isna(best_row["exit_z"]))
        | (focused_days["exit_z"] == best_row["exit_z"])
    )
    & (focused_days["max_hold"] == best_row["max_hold"])
    & (focused_days["gross"] == best_row["gross"])
)

best_days = focused_days[mask].sort_values("day")
display(best_days)

if not focused_trades.empty:
    trade_mask = (
        (focused_trades["candidate"] == best_row["candidate"])
        & (focused_trades["window"] == best_row["window"])
        & (focused_trades["entry_z"] == best_row["entry_z"])
        & (focused_trades["exit_mode"] == best_row["exit_mode"])
        & (
            (focused_trades["exit_z"].isna() & pd.isna(best_row["exit_z"]))
            | (focused_trades["exit_z"] == best_row["exit_z"])
        )
        & (focused_trades["max_hold"] == best_row["max_hold"])
        & (focused_trades["gross"] == best_row["gross"])
    )

    best_trades = focused_trades[trade_mask].sort_values(["day", "entry_idx"])
    display(best_trades)

,candidate,family,kind,note,window,entry_z,exit_z,exit_mode,max_hold,gross,days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
2590,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,NaN,fixed_hold,1500,80,3,443440.0,147813.333333,122160.0,164760.0,17,0.822222,0.8,26084.705882,0.37155,True,692171.111111


,day,trade_count,day_pnl,hit_rate,avg_trade_pnl,median_trade_pnl,window,entry_z,exit_z,exit_mode,max_hold,gross,candidate,family,kind,note
7770,2,6,122160.0,0.833333,20360.000000,20240.0,500,2.25,NaN,fixed_hold,1500,80,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250
7771,3,6,156520.0,0.833333,26086.666667,17480.0,500,2.25,NaN,fixed_hold,1500,80,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250
7772,4,5,164760.0,0.800000,32952.000000,36200.0,500,2.25,NaN,fixed_hold,1500,80,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250


,candidate,family,kind,note,day,window,entry_z,exit_z,exit_mode,max_hold,gross,q_trade,net_position,gross_position,entry_idx,exit_idx,entry_ts,exit_ts,side,entry_z_seen,exit_z_seen,hold,exec_pnl,position
197503,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,2,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,1252,2752,125200,275200,1,-2.258428,-0.761087,1500,11680.0,"[80.0, 0.0, 0.0, 0.0, 0.0]"
197504,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,2,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,2808,4308,280800,430800,1,-2.263723,-2.125230,1500,56360.0,"[80.0, 0.0, 0.0, 0.0, 0.0]"
197505,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,2,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,4368,5868,436800,586800,1,-2.357609,0.876294,1500,36360.0,"[80.0, 0.0, 0.0, 0.0, 0.0]"
197506,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,2,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,6193,7693,619300,769300,-1,2.437243,1.037322,1500,-22720.0,"[-80.0, -0.0, -0.0, -0.0, -0.0]"
197507,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,2,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,7904,9404,790400,940400,1,-2.398069,3.018385,1500,24480.0,"[80.0, 0.0, 0.0, 0.0, 0.0]"
197508,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,2,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,9405,9999,940500,999900,-1,3.006702,-0.338921,594,16000.0,"[-80.0, -0.0, -0.0, -0.0, -0.0]"
197509,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,3,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,929,2429,92900,242900,-1,2.263040,-2.637358,1500,12320.0,"[-80.0, -0.0, -0.0, -0.0, -0.0]"
197510,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,3,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,2430,3930,243000,393000,1,-2.783565,2.113135,1500,-1400.0,"[80.0, 0.0, 0.0, 0.0, 0.0]"
197511,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,3,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,4076,5576,407600,557600,-1,2.346412,0.415596,1500,22000.0,"[-80.0, -0.0, -0.0, -0.0, -0.0]"
197512,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,3,500,2.25,NaN,fixed_hold,1500,80,"[80.0, 0.0, 0.0, 0.0, 0.0]",80.0,80.0,5640,7140,564000,714000,1,-2.574856,1.749237,1500,69640.0,"[80.0, 0.0, 0.0, 0.0, 0.0]"


In [29]:
def make_lead_lag_spec(name_prefix, leader_colour, follower_colour, relation, lag):
    leader = COLOUR_TO_PRODUCT[leader_colour]
    follower = COLOUR_TO_PRODUCT[follower_colour]
    return {
        "name": f"{name_prefix}_lag{lag}",
        "kind": "lead_lag",
        "family": "lead_lag_lag_sweep",
        "leader": leader,
        "follower": follower,
        "relation": relation,
        "lag": int(lag),
        "note": f"{follower_colour} {relation} follows {leader_colour} return over lag {lag}",
    }


LAG_SWEEP_SPECS = []

for lag in [25, 50, 100, 150, 250, 500, 750, 1000, 1250, 1500, 2000]:
    LAG_SWEEP_SPECS.append(make_lead_lag_spec("lead_AMBER_to_RED_inverse", "AMBER", "RED", "inverse", lag))
    LAG_SWEEP_SPECS.append(make_lead_lag_spec("lead_RED_to_YELLOW_inverse", "RED", "YELLOW", "inverse", lag))
    LAG_SWEEP_SPECS.append(make_lead_lag_spec("lead_YELLOW_to_RED_follow", "YELLOW", "RED", "follow", lag))
    LAG_SWEEP_SPECS.append(make_lead_lag_spec("lead_AMBER_to_ORANGE_follow", "AMBER", "ORANGE", "follow", lag))
    LAG_SWEEP_SPECS.append(make_lead_lag_spec("lead_ORANGE_to_YELLOW_follow", "ORANGE", "YELLOW", "follow", lag))

lag_sweep_summary, lag_sweep_days, lag_sweep_trades = run_grid(
    specs=LAG_SWEEP_SPECS,
    windows=[250, 500, 1000, 1500],
    entry_zs=[1.25, 1.50, 1.75, 2.00],
    exit_zs_by_mode={"fixed_hold": [np.nan]},
    max_holds=[500, 1000, 1500, 2500, 4000],
    grosses=[80],
    label="lag_sweep",
)

lag_sweep_summary.to_csv(OUT_DIR / "lag_sweep_summary.csv", index=False)
lag_sweep_days.to_csv(OUT_DIR / "lag_sweep_days.csv", index=False)
lag_sweep_trades.to_csv(OUT_DIR / "lag_sweep_trades.csv", index=False)

display(lag_sweep_summary.head(80))

lag_sweep: total configs = 4400
[   0.00s] lag_sweep 1/4400: lead_AMBER_to_RED_inverse_lag25, w=250, entry=1.25, exit=nan, mode=fixed_hold, hold=500, gross=80
[   1.19s] lag_sweep 250/4400: lead_AMBER_to_ORANGE_follow_lag25, w=250, entry=1.5, exit=nan, mode=fixed_hold, hold=4000, gross=80
[   2.30s] lag_sweep 500/4400: lead_RED_to_YELLOW_inverse_lag50, w=250, entry=2.0, exit=nan, mode=fixed_hold, hold=4000, gross=80
[   3.41s] lag_sweep 750/4400: lead_ORANGE_to_YELLOW_follow_lag50, w=500, entry=1.5, exit=nan, mode=fixed_hold, hold=4000, gross=80
[   4.72s] lag_sweep 1000/4400: lead_YELLOW_to_RED_follow_lag100, w=500, entry=2.0, exit=nan, mode=fixed_hold, hold=4000, gross=80
[   5.84s] lag_sweep 1250/4400: lead_AMBER_to_RED_inverse_lag150, w=1000, entry=1.5, exit=nan, mode=fixed_hold, hold=4000, gross=80
[   7.02s] lag_sweep 1500/4400: lead_AMBER_to_ORANGE_follow_lag150, w=1000, entry=2.0, exit=nan, mode=fixed_hold, hold=4000, gross=80
[   8.15s] lag_sweep 1750/4400: lead_RED_to_YELLOW_

,candidate,family,kind,note,window,entry_z,exit_z,exit_mode,max_hold,gross,days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
2124,lead_RED_to_YELLOW_inverse_lag500,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 500,1000,1.25,NaN,fixed_hold,4000,80,3,415480.0,138493.333333,137760.0,139520.0,8,1.000000,1.000000,51935.000000,0.335804,True,696500.000000
2129,lead_RED_to_YELLOW_inverse_lag500,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 500,1000,1.50,NaN,fixed_hold,4000,80,3,418440.0,139480.000000,135240.0,143400.0,8,1.000000,1.000000,52305.000000,0.342701,True,694420.000000
1979,lead_ORANGE_to_YELLOW_follow_lag250,lead_lag_lag_sweep,lead_lag,YELLOW follow follows ORANGE return over lag 250,1000,2.00,NaN,fixed_hold,4000,80,3,410360.0,136786.666667,125840.0,153800.0,7,1.000000,1.000000,58622.857143,0.374793,True,667540.000000
4160,lead_YELLOW_to_RED_follow_lag2000,lead_lag_lag_sweep,lead_lag,RED follow follows YELLOW return over lag 2000,250,1.25,NaN,fixed_hold,500,80,3,395520.0,131840.000000,122240.0,140400.0,46,0.675000,0.625000,8598.260870,0.354976,True,643462.500000
1717,lead_RED_to_YELLOW_inverse_lag250,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 250,500,2.00,NaN,fixed_hold,1500,80,3,426960.0,142320.000000,105120.0,184160.0,18,0.888889,0.833333,23720.000000,0.431328,True,641811.111111
2904,lead_RED_to_YELLOW_inverse_lag1000,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 1000,500,1.25,NaN,fixed_hold,4000,80,3,386360.0,128786.666667,108440.0,139520.0,8,0.888889,0.666667,48295.000000,0.361114,True,607017.777778
2540,lead_RED_to_YELLOW_inverse_lag750,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 750,1500,1.25,NaN,fixed_hold,500,80,3,415080.0,138360.000000,92960.0,225840.0,36,0.715873,0.600000,11530.000000,0.544088,True,599949.147841
2095,lead_RED_to_YELLOW_inverse_lag500,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 500,250,2.00,NaN,fixed_hold,500,80,3,398840.0,132946.666667,96560.0,163000.0,51,0.627723,0.611111,7820.392157,0.408685,True,595329.417211
1074,lead_AMBER_to_ORANGE_follow_lag100,lead_lag_lag_sweep,lead_lag,ORANGE follow follows AMBER return over lag 100,500,1.75,NaN,fixed_hold,4000,80,3,438280.0,146093.333333,74200.0,214080.0,9,0.888889,0.666667,48697.777778,0.488455,True,590457.777778
2981,lead_YELLOW_to_RED_follow_lag1000,lead_lag_lag_sweep,lead_lag,RED follow follows YELLOW return over lag 1000,500,1.25,NaN,fixed_hold,1000,80,3,448840.0,149613.333333,68120.0,191040.0,27,0.851852,0.777778,16623.703704,0.425631,True,589394.814815


In [30]:
lag_sweep_summary["relationship"] = lag_sweep_summary["candidate"].str.replace(r"_lag\d+$", "", regex=True)
lag_sweep_summary["lag"] = lag_sweep_summary["candidate"].str.extract(r"_lag(\d+)$").astype(int)

best_lag_by_relationship = (
    lag_sweep_summary
    .sort_values(["robust_pass", "robust_score", "total_pnl"], ascending=[False, False, False])
    .groupby("relationship")
    .head(10)
)

display(best_lag_by_relationship[
    [
        "relationship", "lag", "window", "entry_z", "max_hold", "gross",
        "total_pnl", "min_day_pnl", "total_trades", "mean_hit_rate",
        "one_day_dependency", "robust_pass", "robust_score"
    ]
])

,relationship,lag,window,entry_z,max_hold,gross,total_pnl,min_day_pnl,total_trades,mean_hit_rate,one_day_dependency,robust_pass,robust_score
2124,lead_RED_to_YELLOW_inverse,500,1000,1.25,4000,80,415480.0,137760.0,8,1.000000,0.335804,True,696500.000000
2129,lead_RED_to_YELLOW_inverse,500,1000,1.50,4000,80,418440.0,135240.0,8,1.000000,0.342701,True,694420.000000
1979,lead_ORANGE_to_YELLOW_follow,250,1000,2.00,4000,80,410360.0,125840.0,7,1.000000,0.374793,True,667540.000000
4160,lead_YELLOW_to_RED_follow,2000,250,1.25,500,80,395520.0,122240.0,46,0.675000,0.354976,True,643462.500000
1717,lead_RED_to_YELLOW_inverse,250,500,2.00,1500,80,426960.0,105120.0,18,0.888889,0.431328,True,641811.111111
2904,lead_RED_to_YELLOW_inverse,1000,500,1.25,4000,80,386360.0,108440.0,8,0.888889,0.361114,True,607017.777778
2540,lead_RED_to_YELLOW_inverse,750,1500,1.25,500,80,415080.0,92960.0,36,0.715873,0.544088,True,599949.147841
2095,lead_RED_to_YELLOW_inverse,500,250,2.00,500,80,398840.0,96560.0,51,0.627723,0.408685,True,595329.417211
1074,lead_AMBER_to_ORANGE_follow,100,500,1.75,4000,80,438280.0,74200.0,9,0.888889,0.488455,True,590457.777778
2981,lead_YELLOW_to_RED_follow,1000,500,1.25,1000,80,448840.0,68120.0,27,0.851852,0.425631,True,589394.814815


In [31]:
MIX_EXIT_MODES = {
    "fixed_hold": [np.nan],
    "abs_band": [0.25, 0.50, 0.75],
    "zero_cross": [0.0],
}

mix_summary, mix_days, mix_trades = run_grid(
    specs=MIXING_SPECS + PAIRWISE_SPECS,
    windows=[500, 1000, 1500, 2500, 4000],
    entry_zs=[1.25, 1.50, 1.75, 2.00, 2.25],
    exit_zs_by_mode=MIX_EXIT_MODES,
    max_holds=[500, 1000, 1500, 2500, 4000],
    grosses=[40, 60, 80],
    label="mix_pair_sanity",
)

mix_summary.to_csv(OUT_DIR / "mix_pair_sanity_summary.csv", index=False)
mix_days.to_csv(OUT_DIR / "mix_pair_sanity_days.csv", index=False)
mix_trades.to_csv(OUT_DIR / "mix_pair_sanity_trades.csv", index=False)

display(mix_summary.head(80))

mix_pair_sanity: total configs = 11250
[   0.00s] mix_pair_sanity 1/11250: mix_RED_blend_YELLOW_MAGENTA_75_25_mr, w=500, entry=1.25, exit=nan, mode=fixed_hold, hold=500, gross=40
[   2.30s] mix_pair_sanity 250/11250: mix_RED_blend_YELLOW_MAGENTA_75_25_mr, w=500, entry=2.0, exit=0.0, mode=zero_cross, hold=1000, gross=60
[   4.37s] mix_pair_sanity 500/11250: mix_RED_blend_YELLOW_MAGENTA_75_25_mr, w=1000, entry=1.5, exit=0.0, mode=zero_cross, hold=2500, gross=40
[   6.55s] mix_pair_sanity 750/11250: mix_RED_blend_YELLOW_MAGENTA_75_25_mr, w=1000, entry=2.25, exit=0.0, mode=zero_cross, hold=4000, gross=80
[   8.38s] mix_pair_sanity 1000/11250: mix_RED_blend_YELLOW_MAGENTA_75_25_mr, w=1500, entry=2.0, exit=0.0, mode=zero_cross, hold=1000, gross=60
[  10.39s] mix_pair_sanity 1250/11250: mix_RED_blend_YELLOW_MAGENTA_75_25_mr, w=2500, entry=1.5, exit=0.0, mode=zero_cross, hold=2500, gross=40
[  12.47s] mix_pair_sanity 1500/11250: mix_RED_blend_YELLOW_MAGENTA_75_25_mr, w=2500, entry=2.25, exit=0

,candidate,family,kind,note,window,entry_z,exit_z,exit_mode,max_hold,gross,days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
5695,mix_AMBER_blend_YELLOW_ORANGE_50_50_mr,colour_mixing,mix_mr,AMBER residual vs 50% YELLOW + 50% ORANGE,500,1.25,NaN,fixed_hold,4000,80,3,460520.0,153506.666667,112280.0,196020.0,9,1.000000,1.000000,51168.888889,0.425649,True,690580.000000
3145,mix_RED_blend_YELLOW_MAGENTA_50_50_mr,colour_mixing,mix_mr,RED residual vs 50% YELLOW + 50% MAGENTA,2500,1.50,NaN,fixed_hold,4000,80,3,444020.0,148006.666667,88060.0,200500.0,6,0.833333,0.500000,74003.333333,0.451556,True,623056.666667
3070,mix_RED_blend_YELLOW_MAGENTA_50_50_mr,colour_mixing,mix_mr,RED residual vs 50% YELLOW + 50% MAGENTA,2500,1.25,NaN,fixed_hold,4000,80,3,430120.0,143373.333333,81160.0,193500.0,6,0.833333,0.500000,71686.666667,0.449874,True,595356.666667
5690,mix_AMBER_blend_YELLOW_ORANGE_50_50_mr,colour_mixing,mix_mr,AMBER residual vs 50% YELLOW + 50% ORANGE,500,1.25,NaN,fixed_hold,4000,60,3,345390.0,115130.000000,84210.0,147015.0,9,1.000000,1.000000,38376.666667,0.425649,True,519310.000000
190,mix_RED_blend_YELLOW_MAGENTA_75_25_mr,colour_mixing,mix_mr,RED residual vs 75% YELLOW + 25% MAGENTA,500,1.75,NaN,fixed_hold,1500,80,3,366490.0,122163.333333,68990.0,154510.0,20,0.698413,0.571429,18324.500000,0.421594,True,507676.349206
3140,mix_RED_blend_YELLOW_MAGENTA_50_50_mr,colour_mixing,mix_mr,RED residual vs 50% YELLOW + 50% MAGENTA,2500,1.50,NaN,fixed_hold,4000,60,3,333015.0,111005.000000,66045.0,150375.0,6,0.833333,0.500000,55502.500000,0.451556,True,468021.666667
3065,mix_RED_blend_YELLOW_MAGENTA_50_50_mr,colour_mixing,mix_mr,RED residual vs 50% YELLOW + 50% MAGENTA,2500,1.25,NaN,fixed_hold,4000,60,3,322590.0,107530.000000,60870.0,145125.0,6,0.833333,0.500000,53765.000000,0.449874,True,447246.666667
115,mix_RED_blend_YELLOW_MAGENTA_75_25_mr,colour_mixing,mix_mr,RED residual vs 75% YELLOW + 25% MAGENTA,500,1.50,NaN,fixed_hold,1500,80,3,364790.0,121596.666667,31000.0,195470.0,21,0.761905,0.714286,17370.952381,0.535843,True,427158.126998
925,mix_RED_blend_YELLOW_MAGENTA_75_25_mr,colour_mixing,mix_mr,RED residual vs 75% YELLOW + 25% MAGENTA,1500,1.75,NaN,fixed_hold,1000,80,3,277500.0,92500.000000,62330.0,111700.0,19,0.634921,0.571429,14605.263158,0.402523,True,405334.603175
185,mix_RED_blend_YELLOW_MAGENTA_75_25_mr,colour_mixing,mix_mr,RED residual vs 75% YELLOW + 25% MAGENTA,500,1.75,NaN,fixed_hold,1500,60,3,274867.5,91622.500000,51742.5,115882.5,20,0.698413,0.571429,13743.375000,0.421594,True,381558.849206


In [32]:
family_comparison = pd.concat(
    [
        focused_summary.assign(test_set="lead_lag_focused"),
        mix_summary.assign(test_set="mix_pair_sanity"),
    ],
    ignore_index=True,
)

family_table = (
    family_comparison
    .groupby(["test_set", "family"])
    .agg(
        best_total_pnl=("total_pnl", "max"),
        best_min_day_pnl=("min_day_pnl", "max"),
        best_robust_score=("robust_score", "max"),
        robust_count=("robust_pass", "sum"),
        configs=("candidate", "size"),
    )
    .sort_values("best_robust_score", ascending=False)
)

display(family_table)

best_total_pnl  best_min_day_pnl  best_robust_score  robust_count  configs
test_set         family                                                                                   
lead_lag_focused lead_lag             448840.0          122160.0      692171.111111          5394    20625
mix_pair_sanity  colour_mixing        460520.0          112280.0      690580.000000            78     7500
                 pairwise             150900.0           26040.0      204604.356339             9     3750

In [33]:
def candidate_card(summary_df, n=20):
    show_cols = [
        "candidate", "family", "kind", "note",
        "window", "entry_z", "exit_mode", "exit_z", "max_hold", "gross",
        "total_pnl", "mean_day_pnl", "min_day_pnl", "max_day_pnl",
        "total_trades", "mean_hit_rate", "min_hit_rate",
        "pnl_per_trade",
        "one_day_dependency", "robust_pass", "robust_score",
    ]
    return summary_df.sort_values(
        ["robust_pass", "robust_score", "min_day_pnl", "total_pnl"],
        ascending=[False, False, False, False],
    )[show_cols].head(n)


final_candidates = pd.concat(
    [
        focused_summary.assign(test_set="focused_lead_lag"),
        lag_sweep_summary.assign(test_set="lag_sweep"),
        mix_summary.assign(test_set="mix_pair_sanity"),
    ],
    ignore_index=True,
)

final_candidates = final_candidates.sort_values(
    ["robust_pass", "robust_score", "min_day_pnl", "total_pnl"],
    ascending=[False, False, False, False],
)

final_candidates.to_csv(OUT_DIR / "final_uv_visor_candidates_all.csv", index=False)

display(candidate_card(final_candidates, n=50))

,candidate,family,kind,note,window,entry_z,exit_mode,exit_z,max_hold,gross,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
20625,lead_RED_to_YELLOW_inverse_lag500,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 500,1000,1.25,fixed_hold,NaN,4000,80,415480.0,138493.333333,137760.0,139520.0,8,1.000000,1.000000,51935.000000,0.335804,True,696500.000000
20626,lead_RED_to_YELLOW_inverse_lag500,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 500,1000,1.50,fixed_hold,NaN,4000,80,418440.0,139480.000000,135240.0,143400.0,8,1.000000,1.000000,52305.000000,0.342701,True,694420.000000
0,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,fixed_hold,NaN,1500,80,443440.0,147813.333333,122160.0,164760.0,17,0.822222,0.800000,26084.705882,0.371550,True,692171.111111
25025,mix_AMBER_blend_YELLOW_ORANGE_50_50_mr,colour_mixing,mix_mr,AMBER residual vs 50% YELLOW + 50% ORANGE,500,1.25,fixed_hold,NaN,4000,80,460520.0,153506.666667,112280.0,196020.0,9,1.000000,1.000000,51168.888889,0.425649,True,690580.000000
20627,lead_ORANGE_to_YELLOW_follow_lag250,lead_lag_lag_sweep,lead_lag,YELLOW follow follows ORANGE return over lag 250,1000,2.00,fixed_hold,NaN,4000,80,410360.0,136786.666667,125840.0,153800.0,7,1.000000,1.000000,58622.857143,0.374793,True,667540.000000
20628,lead_YELLOW_to_RED_follow_lag2000,lead_lag_lag_sweep,lead_lag,RED follow follows YELLOW return over lag 2000,250,1.25,fixed_hold,NaN,500,80,395520.0,131840.000000,122240.0,140400.0,46,0.675000,0.625000,8598.260870,0.354976,True,643462.500000
1,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.00,fixed_hold,NaN,1500,80,426960.0,142320.000000,105120.0,184160.0,18,0.888889,0.833333,23720.000000,0.431328,True,641811.111111
20629,lead_RED_to_YELLOW_inverse_lag250,lead_lag_lag_sweep,lead_lag,YELLOW inverse follows RED return over lag 250,500,2.00,fixed_hold,NaN,1500,80,426960.0,142320.000000,105120.0,184160.0,18,0.888889,0.833333,23720.000000,0.431328,True,641811.111111
2,lead_RED_to_YELLOW_inverse_lag250,lead_lag,lead_lag,YELLOW inversely follows RED return over lag 250,500,2.25,fixed_hold,NaN,1500,72,399096.0,133032.000000,109944.0,148284.0,17,0.822222,0.800000,23476.235294,0.371550,True,623395.111111
25026,mix_RED_blend_YELLOW_MAGENTA_50_50_mr,colour_mixing,mix_mr,RED residual vs 50% YELLOW + 50% MAGENTA,2500,1.50,fixed_hold,NaN,4000,80,444020.0,148006.666667,88060.0,200500.0,6,0.833333,0.500000,74003.333333,0.451556,True,623056.666667
